# MediMultiBERT-KCLT: full experimental pipeline (Colab)



In [ ]:
# ====== SETTINGS ======
MODE = "pilot"          # "pilot" or "full"
KG_SOURCE = "doid"      # "doid" (open, interim) or "umls"
UMLS_DIR = "/content/drive/MyDrive/UMLS/META"   # folder with MRCONSO.RRF, MRREL.RRF, MRSTY.RRF
SEEDS = [42, 101, 202, 303, 404]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/KCLT'
os.makedirs(WORK, exist_ok=True); os.chdir(WORK); print(os.getcwd())
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
/content/drive/MyDrive/KCLT
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Write the code files

In [ ]:
%%writefile common.py
"""Shared constants and helpers for the MediMultiBERT-KCLT pipeline."""
import os, random, json, hashlib
import numpy as np

LANGS = ["en", "es", "fr", "hi", "ta"]
LANG_NAMES = {"en": "English", "es": "Spanish", "fr": "French", "hi": "Hindi", "ta": "Tamil"}

# The 24 Symptom2Disease labels, mapped to the concept name used to find the disease node in the KG.
DISEASE_CONCEPTS = {
    "Psoriasis": "Psoriasis",
    "Varicose Veins": "Varicose veins",
    "peptic ulcer disease": "Peptic ulcer",
    "drug reaction": "Adverse reaction to drug",
    "gastroesophageal reflux disease": "Gastroesophageal reflux disease",
    "allergy": "Hypersensitivity",
    "urinary tract infection": "Urinary tract infection",
    "Malaria": "Malaria",
    "Jaundice": "Jaundice",
    "Cervical spondylosis": "Cervical spondylosis",
    "Migraine": "Migraine",
    "Hypertension": "Hypertension",
    "Bronchial Asthma": "Asthma",
    "Acne": "Acne",
    "Arthritis": "Arthritis",
    "Dimorphic Hemorrhoids": "Hemorrhoids",
    "Pneumonia": "Pneumonia",
    "Common Cold": "Common cold",
    "Fungal infection": "Mycosis",
    "Dengue": "Dengue fever",
    "Impetigo": "Impetigo",
    "Chicken pox": "Chickenpox",
    "Typhoid": "Typhoid fever",
    "diabetes": "Diabetes mellitus",
}


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


def text_group_id(text: str) -> str:
    """Group key for a source narrative: identical (normalised) English texts share one group,
    so exact duplicates in Symptom2Disease can never be split across train/val/test."""
    norm = " ".join(str(text).lower().split())
    return hashlib.md5(norm.encode("utf-8")).hexdigest()[:12]


def save_json(obj, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

# Alternative names used to find each disease in non-UMLS sources (e.g. the Disease Ontology)
DISEASE_ALIASES = {
    "drug reaction": ["drug allergy", "drug hypersensitivity"],
    "allergy": ["allergic disease"],
    "Fungal infection": ["fungal infectious disease"],
    "Dimorphic Hemorrhoids": ["hemorrhoid"],
    "Dengue": ["dengue disease"],
    "peptic ulcer disease": ["peptic ulcer disease"],
    "Jaundice": ["jaundice", "hyperbilirubinemia"],
    "Cervical spondylosis": ["spondylosis", "cervical spondylosis"],
    "Bronchial Asthma": ["asthma"],
    "Typhoid": ["typhoid fever"],
    "diabetes": ["diabetes mellitus"],
    "Chicken pox": ["chickenpox"],
}


Overwriting common.py


In [ ]:
%%writefile s01_translate.py
"""
Step 1 - Build the multilingual corpus.

Translates every English Symptom2Disease narrative into Spanish, French, Hindi and Tamil
(default: offline NLLB-200 model, facebook/nllb-200-distilled-600M; optional: --engine google),
keeps the source identifier on every translation, and exports a random 10% review sheet
per language for native-speaker checking.

Usage
  python s01_translate.py --input Symptom2Disease.csv --out data
  # after reviewers fill in review/review_<lang>.csv (column 'corrected_text', 'status'):
  python s01_translate.py --apply_review --out data

Output
  data/multilingual.csv  columns: uid, source_id, group_id, language, label, text
"""
import argparse, os, time
import pandas as pd
from tqdm import tqdm
from common import LANGS, text_group_id

TARGETS = [l for l in LANGS if l != "en"]


NLLB_CODES = {"es": "spa_Latn", "fr": "fra_Latn", "hi": "hin_Deva", "ta": "tam_Taml"}
NLLB_MODEL = "facebook/nllb-200-distilled-600M"
_nllb = {}


def _load_cache(path):
    if os.path.exists(path):
        c = pd.read_csv(path).dropna()
        return dict(zip(c["src"], c["tgt"]))
    return {}


def _save_cache(cache, path):
    pd.DataFrame({"src": list(cache), "tgt": list(cache.values())}).to_csv(path, index=False)


def translate_nllb(texts, target, cache_path, bs=16, beams=4):
    """Offline neural MT with NLLB-200 (runs on the local GPU; no external API)."""
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    cache = _load_cache(cache_path)
    todo = [t for t in dict.fromkeys(texts) if t not in cache]
    if todo:
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        if "model" not in _nllb:
            _nllb["tok"] = AutoTokenizer.from_pretrained(NLLB_MODEL, src_lang="eng_Latn")
            m = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL)
            _nllb["model"] = (m.half() if dev == "cuda" else m).to(dev).eval()
        tok, model = _nllb["tok"], _nllb["model"]
        tgt_id = tok.convert_tokens_to_ids(NLLB_CODES[target])
        for i in tqdm(range(0, len(todo), bs), desc=f"en->{target} (NLLB)"):
            batch = todo[i:i + bs]
            enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(dev)
            with torch.no_grad():
                gen = model.generate(**enc, forced_bos_token_id=tgt_id, num_beams=beams, max_new_tokens=300)
            for src, out in zip(batch, tok.batch_decode(gen, skip_special_tokens=True)):
                cache[src] = out
            if (i // bs) % 10 == 0:
                _save_cache(cache, cache_path)
        _save_cache(cache, cache_path)
    return [cache[t] for t in texts]


def translate_google(texts, target, cache_path, sleep=0.2, retries=4):
    """Google Translate via deep-translator (often blocked from shared cloud IPs such as Colab)."""
    from deep_translator import GoogleTranslator
    cache = _load_cache(cache_path)
    tr = GoogleTranslator(source="en", target=target)
    out = []
    for t in tqdm(texts, desc=f"en->{target} (Google)"):
        if t not in cache:
            err = None
            for k in range(retries):
                try:
                    cache[t] = tr.translate(t); break
                except Exception as e:
                    err = e; time.sleep(2 ** k)
            if t not in cache:
                raise RuntimeError(f"Google translation failed ({type(err).__name__}: {err}). Use --engine nllb.")
            if len(cache) % 50 == 0:
                _save_cache(cache, cache_path)
            time.sleep(sleep)
        out.append(cache[t])
    _save_cache(cache, cache_path)
    return out


def translate_list(texts, target, cache_path, engine="nllb"):
    return (translate_nllb if engine == "nllb" else translate_google)(texts, target, cache_path)


def build(args):
    os.makedirs(args.out, exist_ok=True); os.makedirs(f"{args.out}/cache", exist_ok=True)
    df = pd.read_csv(args.input)
    df = df[["label", "text"]].reset_index(drop=True)
    df["source_id"] = df.index                      # one id per original English row
    df["group_id"] = df["text"].map(text_group_id)  # duplicates share a group
    rows = [df.assign(language="en")]
    for lang in TARGETS:
        t = translate_list(df["text"].tolist(), lang, f"{args.out}/cache/{args.engine}_{lang}.csv", args.engine)
        rows.append(df.assign(language=lang, text=t))
    ml = pd.concat(rows, ignore_index=True)
    ml["uid"] = ml["language"] + "_" + ml["source_id"].astype(str)
    ml = ml[["uid", "source_id", "group_id", "language", "label", "text"]]
    ml.to_csv(f"{args.out}/multilingual.csv", index=False)
    print(f"Wrote {len(ml)} rows ->", f"{args.out}/multilingual.csv")

    # 10% review sample per language (stratified by label), for native-speaker checking
    os.makedirs(f"{args.out}/review", exist_ok=True)
    for lang in TARGETS:
        sub = ml[ml.language == lang]
        samp = sub.groupby("label").sample(frac=args.review_frac, random_state=42)
        src = df.set_index("source_id")["text"]
        samp = samp.assign(english_source=samp["source_id"].map(src),
                           status="", corrected_text="", reviewer="")
        samp.to_csv(f"{args.out}/review/review_{lang}.csv", index=False)
        print(f"  review sheet: review_{lang}.csv ({len(samp)} rows)")


def apply_review(args):
    ml = pd.read_csv(f"{args.out}/multilingual.csv")
    log = []
    for lang in TARGETS:
        p = f"{args.out}/review/review_{lang}.csv"
        if not os.path.exists(p):
            continue
        r = pd.read_csv(p).fillna("")
        fix = r[r["corrected_text"].str.strip() != ""]
        for _, row in fix.iterrows():
            ml.loc[ml.uid == row["uid"], "text"] = row["corrected_text"]
        log.append({"language": lang, "reviewed": len(r), "corrected": len(fix)})
    ml.to_csv(f"{args.out}/multilingual.csv", index=False)
    pd.DataFrame(log).to_csv(f"{args.out}/review/review_summary.csv", index=False)
    print(pd.DataFrame(log))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", default="Symptom2Disease.csv")
    ap.add_argument("--out", default="data")
    ap.add_argument("--review_frac", type=float, default=0.10)
    ap.add_argument("--engine", choices=["nllb", "google"], default="nllb")
    ap.add_argument("--apply_review", action="store_true")
    a = ap.parse_args()
    apply_review(a) if a.apply_review else build(a)


Overwriting s01_translate.py


In [ ]:
%%writefile s02_split.py
"""
Step 2 - Leakage-safe split + audit.

Splits at the level of source-narrative GROUPS (identical English texts form one group),
stratified by disease label, 70/15/15. All five language versions of a narrative, and all
duplicates of it, land in the same subset. Then audits the result.

Usage:  python s02_split.py --data data/multilingual.csv --out data --seed 42
Output: data/train.csv, data/val.csv, data/test.csv, data/split_audit.json
"""
import argparse
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from common import save_json


def main(a):
    ml = pd.read_csv(a.data)
    g = ml[ml.language == "en"].groupby("group_id")["label"].first().reset_index()
    tr_g, tmp_g = train_test_split(g, test_size=0.30, stratify=g.label, random_state=a.seed)
    va_g, te_g = train_test_split(tmp_g, test_size=0.50, stratify=tmp_g.label, random_state=a.seed)
    parts = {"train": set(tr_g.group_id), "val": set(va_g.group_id), "test": set(te_g.group_id)}
    out = {}
    for name, gs in parts.items():
        d = ml[ml.group_id.isin(gs)].reset_index(drop=True)
        d.to_csv(f"{a.out}/{name}.csv", index=False); out[name] = d

    # ---------------- audit ----------------
    audit = {"sizes": {k: len(v) for k, v in out.items()},
             "source_groups": {k: v.group_id.nunique() for k, v in out.items()},
             "per_language": {k: v.language.value_counts().to_dict() for k, v in out.items()},
             "test_per_class": out["test"].label.value_counts().to_dict()}
    names = list(out)
    audit["shared_groups"] = {f"{x}&{y}": len(set(out[x].group_id) & set(out[y].group_id))
                              for i, x in enumerate(names) for y in names[i + 1:]}
    audit["shared_source_ids"] = {f"{x}&{y}": len(set(out[x].source_id) & set(out[y].source_id))
                                  for i, x in enumerate(names) for y in names[i + 1:]}
    norm = lambda s: set(s.str.lower().str.split().str.join(" "))
    audit["identical_texts"] = {f"{x}&{y}": len(norm(out[x].text) & norm(out[y].text))
                                for i, x in enumerate(names) for y in names[i + 1:]}
    nd = {}
    for lang in sorted(ml.language.unique()):
        tr = out["train"][out["train"].language == lang].text
        te = out["test"][out["test"].language == lang].text
        v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(pd.concat([tr, te]))
        m = cosine_similarity(v.transform(te), v.transform(tr)).max(axis=1)
        nd[lang] = int((m >= a.near_dup).sum())
    audit[f"near_duplicates_test_vs_train_cos>={a.near_dup}"] = nd
    save_json(audit, f"{a.out}/split_audit.json")
    for k, v in audit.items():
        print(f"{k}: {v}")
    assert all(v == 0 for v in audit["shared_groups"].values()), "LEAKAGE: shared groups"


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/multilingual.csv")
    ap.add_argument("--out", default="data")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--near_dup", type=float, default=0.9)
    main(ap.parse_args())


Overwriting s02_split.py


In [ ]:
%%writefile s03_build_kg.py
"""
Step 3 - Build the medical knowledge graph.

MODE A (recommended, matches the paper): UMLS Metathesaurus (includes SNOMED CT US).
  Requires a free UMLS licence (https://uts.nlm.nih.gov). Download the UMLS "Full Release"
  and point --umls_dir at the folder containing MRCONSO.RRF, MRREL.RRF and MRSTY.RRF.

  python s03_build_kg.py --umls_dir /path/to/2025AA/META --out kg

MODE B (no UMLS licence yet / open alternative): a CSV of triples
  head,relation,tail     e.g.  Dengue fever,has_symptom,Fever
  python s03_build_kg.py --triples_csv my_triples.csv --out kg
  Triples must come from an external source (ontology, textbook, curated list),
  NEVER from the Symptom2Disease narratives themselves, otherwise labels leak into the KG.

Output
  kg/nodes.csv   nid, cui, name, synonyms ('|'-joined; ENG/SPA/FRE where available), is_disease, label
  kg/edges.csv   src, dst, rel      (undirected; both directions are added at load time)
  kg/seed_map.csv  the UMLS concept chosen for each of the 24 diseases - CHECK THIS BY HAND
  kg/kg_stats.json
"""
import argparse, os, collections
import pandas as pd
from common import DISEASE_CONCEPTS, DISEASE_ALIASES, save_json

# Semantic types kept as nodes (UMLS TUIs)
TUI_KEEP = {"T047", "T184", "T033", "T046", "T037", "T019", "T020", "T191", "T048", "T190",
            "T023", "T029", "T031", "T121", "T061"}
TUI_SYMPTOM = {"T184"}
REL_KEEP = {"PAR", "CHD", "RB", "RN", "RO"}
RELA_CLINICAL = {"has_manifestation", "manifestation_of", "has_associated_finding",
                 "associated_finding_of", "has_finding_site", "finding_site_of", "cause_of",
                 "due_to", "associated_with", "has_associated_morphology",
                 "associated_morphology_of", "may_treat", "may_be_treated_by", "clinically_associated_with",
                 "has_sign_or_symptom", "sign_or_symptom_of", "disease_has_finding", "is_finding_of_disease"}
LANG_KEEP = {"ENG", "SPA", "FRE"}
CONSO_COLS = ["CUI", "LAT", "TS", "LUI", "STT", "SUI", "ISPREF", "AUI", "SAUI", "SCUI", "SDUI",
              "SAB", "TTY", "CODE", "STR", "SRL", "SUPPRESS", "CVF"]
REL_COLS = ["CUI1", "AUI1", "STYPE1", "REL", "CUI2", "AUI2", "STYPE2", "RELA", "RUI", "SRUI",
            "SAB", "SL", "RG", "DIR", "SUPPRESS", "CVF"]


def rrf(path, cols, usecols, chunksize=2_000_000):
    return pd.read_csv(
        path, sep="|", header=None, names=cols + ["_"], usecols=usecols, dtype=str,
        quoting=3, chunksize=chunksize, na_filter=False, encoding="utf-8")


def build_umls(a):
    D = a.umls_dir
    print("[1/4] MRSTY: semantic types")
    sty = pd.read_csv(f"{D}/MRSTY.RRF", sep="|", header=None, usecols=[0, 1], names=["CUI", "TUI"],
                      dtype=str, quoting=3)
    keep_cui = set(sty[sty.TUI.isin(TUI_KEEP)].CUI)
    symptom_cui = set(sty[sty.TUI.isin(TUI_SYMPTOM)].CUI)

    print("[2/4] MRCONSO pass 1: locate the 24 disease concepts + symptom names")
    want = {v.lower(): k for k, v in DISEASE_CONCEPTS.items()}
    seed_hits = collections.defaultdict(collections.Counter)
    sym_pref = {}
    for ch in rrf(f"{D}/MRCONSO.RRF", CONSO_COLS, ["CUI", "LAT", "TS", "STT", "ISPREF", "SAB", "STR", "SUPPRESS"]):
        ch = ch[(ch.LAT == "ENG") & (ch.SUPPRESS == "N") & ch.CUI.isin(keep_cui)]
        low = ch.STR.str.lower()
        hit = ch[low.isin(want)]
        for cui, s, ts in zip(hit.CUI, hit.STR.str.lower(), hit.TS):
            seed_hits[want[s]][cui] += 2 if ts == "P" else 1
        sp = ch[ch.CUI.isin(symptom_cui) & (ch.TS == "P") & (ch.STT == "PF") & (ch.ISPREF == "Y")
                & ch.SAB.isin(["SNOMEDCT_US", "MSH", "MDR", "HPO"])]
        for cui, s in zip(sp.CUI, sp.STR):
            if len(s.split()) <= 4:
                sym_pref.setdefault(cui, s)
    seed = {}
    for label in DISEASE_CONCEPTS:
        if a.seed_map and os.path.exists(a.seed_map):
            break
        if seed_hits[label]:
            seed[label] = seed_hits[label].most_common(1)[0][0]
        else:
            print(f"  WARNING: no UMLS match for '{label}' - add it to a --seed_map CSV")
    if a.seed_map and os.path.exists(a.seed_map):
        sm = pd.read_csv(a.seed_map); seed = dict(zip(sm.label, sm.cui))
    seed_cuis = set(seed.values())

    print("[3/4] MRREL: relations")
    edges = []
    for ch in rrf(f"{D}/MRREL.RRF", REL_COLS, ["CUI1", "REL", "CUI2", "RELA", "SUPPRESS"]):
        ch = ch[(ch.SUPPRESS == "N") & ch.REL.isin(REL_KEEP) & (ch.CUI1 != ch.CUI2)
                & ch.CUI1.isin(keep_cui) & ch.CUI2.isin(keep_cui)]
        ch = ch[(ch.REL != "RO") | ch.RELA.isin(RELA_CLINICAL)]
        edges.append(ch[["CUI1", "CUI2", "REL", "RELA"]])
    E = pd.concat(edges, ignore_index=True)
    # node set: diseases + 1-hop neighbours of diseases + symptom concepts
    hop1 = set(E[E.CUI1.isin(seed_cuis)].CUI2) | set(E[E.CUI2.isin(seed_cuis)].CUI1)
    if a.max_hop1 and len(hop1) > a.max_hop1:   # keep clinically-typed neighbours first
        rel_n = E[(E.CUI1.isin(seed_cuis) | E.CUI2.isin(seed_cuis)) & (E.RELA != "")]
        pri = (set(rel_n.CUI1) | set(rel_n.CUI2)) & hop1
        hop1 = pri | set(list(hop1 - pri)[: max(0, a.max_hop1 - len(pri))])
    nodes = seed_cuis | hop1 | set(sym_pref)
    E = E[E.CUI1.isin(nodes) & E.CUI2.isin(nodes)]
    E["key"] = [tuple(sorted(x)) for x in zip(E.CUI1, E.CUI2)]
    E = E.drop_duplicates("key")
    deg = collections.Counter(list(E.CUI1) + list(E.CUI2))
    nodes = {c for c in nodes if deg[c] > 0 or c in seed_cuis}

    print("[4/4] MRCONSO pass 2: names and ENG/SPA/FRE synonyms")
    pref, syn = {}, collections.defaultdict(list)
    for ch in rrf(f"{D}/MRCONSO.RRF", CONSO_COLS, ["CUI", "LAT", "TS", "STT", "ISPREF", "STR", "SUPPRESS"]):
        ch = ch[ch.CUI.isin(nodes) & ch.LAT.isin(LANG_KEEP) & (ch.SUPPRESS == "N")]
        for cui, lat, ts, stt, isp, s in zip(ch.CUI, ch.LAT, ch.TS, ch.STT, ch.ISPREF, ch.STR):
            if lat == "ENG" and ts == "P" and stt == "PF" and isp == "Y":
                pref.setdefault(cui, s)
            if len(syn[cui]) < a.max_syn and s not in syn[cui] and len(s) < 80:
                syn[cui].append(s)
    write_kg(a.out, nodes, pref, syn, E[["CUI1", "CUI2", "REL"]].values.tolist(), seed)


def build_triples(a):
    T = pd.read_csv(a.triples_csv).dropna()
    names = sorted(set(T["head"]) | set(T["tail"]))
    cui = {n: f"N{i:06d}" for i, n in enumerate(names)}
    low = {n.lower(): n for n in names}
    seed = {}
    for label, concept in DISEASE_CONCEPTS.items():   # first matching name wins: concept, label, aliases
        for cand in [concept, label] + DISEASE_ALIASES.get(label, []):
            if cand.lower() in low:
                seed[label] = cui[low[cand.lower()]]; break
    missing = [l for l in DISEASE_CONCEPTS if l not in seed]
    for l in missing:  # disease without triples still gets an (isolated) node
        n = DISEASE_CONCEPTS[l]; cui[n] = f"D{len(cui):06d}"; seed[l] = cui[n]; names.append(n)
    pref = {cui[n]: n for n in names}
    syn = {cui[n]: [n] for n in names}
    E = [[cui[h], cui[t], r] for h, r, t in zip(T["head"], T["relation"], T["tail"])]
    write_kg(a.out, set(pref), pref, syn, E, seed)


def write_kg(out, nodes, pref, syn, edges, seed):
    os.makedirs(out, exist_ok=True)
    nodes = sorted(c for c in nodes if c in pref or syn.get(c))
    nid = {c: i for i, c in enumerate(nodes)}
    lab_of = {c: l for l, c in seed.items()}
    N = pd.DataFrame({"nid": range(len(nodes)), "cui": nodes,
                      "name": [pref.get(c, (syn.get(c) or [c])[0]) for c in nodes],
                      "synonyms": ["|".join(syn.get(c, [])) for c in nodes],
                      "is_disease": [int(c in lab_of) for c in nodes],
                      "label": [lab_of.get(c, "") for c in nodes]})
    Ed = pd.DataFrame([(nid[s], nid[d], r) for s, d, r in edges if s in nid and d in nid],
                      columns=["src", "dst", "rel"]).drop_duplicates()
    N.to_csv(f"{out}/nodes.csv", index=False); Ed.to_csv(f"{out}/edges.csv", index=False)
    pd.DataFrame([{"label": l, "cui": c, "name": pref.get(c, "")} for l, c in seed.items()]) \
        .to_csv(f"{out}/seed_map.csv", index=False)
    stats = {"nodes": len(N), "edges": len(Ed), "disease_nodes": int(N.is_disease.sum()),
             "diseases_mapped": len(seed), "edge_types": Ed.rel.value_counts().to_dict()}
    save_json(stats, f"{out}/kg_stats.json"); print(stats)
    print("CHECK kg/seed_map.csv by hand: each of the 24 labels must map to the right concept.")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--umls_dir"); ap.add_argument("--triples_csv")
    ap.add_argument("--seed_map", help="optional CSV label,cui to override the automatic disease mapping")
    ap.add_argument("--out", default="kg")
    ap.add_argument("--max_hop1", type=int, default=3000)
    ap.add_argument("--max_syn", type=int, default=12)
    a = ap.parse_args()
    if a.umls_dir: build_umls(a)
    elif a.triples_csv: build_triples(a)
    else: ap.error("give --umls_dir or --triples_csv")


Overwriting s03_build_kg.py


In [ ]:
%%writefile s03b_doid_triples.py
"""
Interim open knowledge source (CC0): Human Disease Ontology (doid.obo).
Writes triples (head, relation, tail) for s03_build_kg.py --triples_csv:
  is_a          disease -> parent disease          (hierarchy)
  has_symptom   disease -> symptom phrase           (parsed from the curated DO definitions)
  synonym       disease -> exact synonym
Use for the pilot only; the final experiments should use UMLS (s03_build_kg.py --umls_dir).

python s03b_doid_triples.py --obo doid.obo --out doid_triples.csv
"""
import argparse, re
import pandas as pd


def parse(path):
    terms, cur = [], None
    for line in open(path, encoding="utf-8"):
        line = line.rstrip("\n")
        if line == "[Term]":
            cur = {"syn": [], "isa": [], "def": ""}; terms.append(cur); continue
        if line.startswith("["):
            cur = None; continue
        if cur is None or ": " not in line:
            continue
        k, v = line.split(": ", 1)
        if k == "id": cur["id"] = v
        elif k == "name": cur["name"] = v
        elif k == "synonym":
            m = re.match(r'"(.*)" EXACT', v)
            if m: cur["syn"].append(m.group(1))
        elif k == "is_a": cur["isa"].append(v.split(" ")[0])
        elif k == "def": cur["def"] = v
        elif k == "is_obsolete": cur["obs"] = True
    return [t for t in terms if "id" in t and "name" in t and not t.get("obs")]


def main(a):
    T = parse(a.obo); name = {t["id"]: t["name"] for t in T}
    rows = []
    for t in T:
        for p in t["isa"]:
            if p in name: rows.append((t["name"], "is_a", name[p]))
        for s in re.findall(r"has_symptom ([^,\.;]+?)(?=,| and |\.|;|$)", t["def"]):
            s = re.sub(r"\(.*?\)", "", s).strip()
            if 2 < len(s) < 60: rows.append((t["name"], "has_symptom", s))
        for s in t["syn"][:5]:
            rows.append((t["name"], "synonym", s))
    df = pd.DataFrame(rows, columns=["head", "relation", "tail"]).drop_duplicates()
    df.to_csv(a.out, index=False); print(df.relation.value_counts())


if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--obo", default="doid.obo"); ap.add_argument("--out", default="doid_triples.csv")
    main(ap.parse_args())


Overwriting s03b_doid_triples.py


In [ ]:
%%writefile s04_link_concepts.py
"""
Step 4 - Link every narrative to knowledge-graph concepts and extract its subgraph.

Language-agnostic linking: every word n-gram (1-3 words) of the narrative and every KG concept
name/synonym is embedded with a multilingual UMLS-aligned encoder (SapBERT-XLMR by default).
Concepts whose best span similarity exceeds a threshold become seed nodes. The per-narrative
subgraph = seed nodes + their 1-hop KG neighbours (+ optionally the 24 disease nodes, which are
identical for every narrative and therefore carry no label information).

Linking uses only the narrative text, never its label.

Usage: python s04_link_concepts.py --data data --kg kg --out kg
Output: kg/node_emb.pt   (N x d node features, frozen GAT input)
        kg/graphs.pt     {uid: {"nodes": LongTensor, "edge_index": LongTensor[2,E], "seed_sim": FloatTensor}}
        kg/linking_stats.json
"""
import argparse, re, collections
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from common import save_json

PUNCT = re.compile(r"[\.,;:!\?\(\)\"'“”‘’\[\]{}।]")


@torch.no_grad()
def embed(texts, tok, enc, device, bs=256, max_len=32):
    out = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i:i + bs], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
        h = enc(**b).last_hidden_state[:, 0]            # [CLS], as in SapBERT
        out.append(F.normalize(h.float(), dim=-1).cpu())
    return torch.cat(out) if out else torch.zeros(0, enc.config.hidden_size)


def spans(text, n_max=3):
    w = PUNCT.sub(" ", str(text)).split()
    return list({" ".join(w[i:i + n]) for n in range(1, n_max + 1) for i in range(len(w) - n + 1)})


def main(a):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(a.linker); enc = AutoModel.from_pretrained(a.linker).to(dev).eval()
    if dev.type == "cuda": enc.half()

    N = pd.read_csv(f"{a.kg}/nodes.csv").fillna("")
    E = pd.read_csv(f"{a.kg}/edges.csv")
    # node embedding = mean of normalised name + synonym embeddings
    names, owner = [], []
    for nid, nm, sy in zip(N.nid, N.name, N.synonyms):
        for s in list(dict.fromkeys([nm] + [x for x in sy.split("|") if x]))[: a.max_syn]:
            names.append(s); owner.append(nid)
    print(f"Embedding {len(names)} concept strings for {len(N)} nodes")
    Z = embed(names, tok, enc, dev)
    owner = torch.tensor(owner)
    node_emb = torch.zeros(len(N), Z.shape[1]).index_add_(0, owner, Z)
    node_emb = F.normalize(node_emb / torch.bincount(owner, minlength=len(N)).clamp(min=1)[:, None], dim=-1)
    torch.save(node_emb.half(), f"{a.out}/node_emb.pt")

    adj = collections.defaultdict(set)
    for s, d in zip(E.src, E.dst):
        adj[s].add(d); adj[d].add(s)
    disease_nodes = N[N.is_disease == 1].nid.tolist() if a.add_disease_nodes else []

    rows = pd.concat([pd.read_csv(f"{a.data}/{s}.csv") for s in ("train", "val", "test")])
    graphs, n_seeds, n_nodes, by_lang = {}, [], [], collections.defaultdict(list)
    NE = node_emb.float().to(dev)
    for uid, text, lang in tqdm(list(zip(rows.uid, rows.text, rows.language)), desc="linking"):
        sp = spans(text)
        S = embed(sp, tok, enc, dev, max_len=16).to(dev) if sp else torch.zeros(0, NE.shape[1], device=dev)
        sim = (S @ NE.T).max(0).values if len(sp) else torch.zeros(len(N), device=dev)
        top = torch.topk(sim, min(a.top_k, len(N)))
        keep = [(int(i), float(v)) for v, i in zip(top.values, top.indices) if v >= a.threshold]
        if len(keep) < a.min_seeds:   # always keep a few best concepts
            keep = [(int(i), float(v)) for v, i in zip(top.values[: a.min_seeds], top.indices[: a.min_seeds])]
        seeds = [i for i, _ in keep]
        nb = []
        for i in seeds:  # 1-hop neighbours, ranked by similarity to the narrative
            cand = sorted(adj[i], key=lambda j: -float(sim[j]))[: a.max_nb_per_seed]
            nb.extend(cand)
        sel = list(dict.fromkeys(seeds + nb + disease_nodes))[: a.max_nodes + len(disease_nodes)]
        loc = {g: k for k, g in enumerate(sel)}
        ei = [(loc[s], loc[d]) for s in sel for d in adj[s] if d in loc]
        ei = torch.tensor(ei, dtype=torch.long).T if ei else torch.zeros(2, 0, dtype=torch.long)
        seed_sim = torch.tensor([float(sim[g]) for g in sel])
        graphs[uid] = {"nodes": torch.tensor(sel, dtype=torch.long), "edge_index": ei, "seed_sim": seed_sim}
        n_seeds.append(len(seeds)); n_nodes.append(len(sel)); by_lang[lang].append(float(top.values[0]))
    torch.save(graphs, f"{a.out}/graphs.pt")
    stats = {"narratives": len(graphs), "mean_seed_concepts": float(np.mean(n_seeds)),
             "mean_subgraph_nodes": float(np.mean(n_nodes)),
             "mean_best_similarity_by_language": {k: float(np.mean(v)) for k, v in by_lang.items()},
             "threshold": a.threshold, "linker": a.linker}
    save_json(stats, f"{a.out}/linking_stats.json"); print(stats)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data"); ap.add_argument("--kg", default="kg"); ap.add_argument("--out", default="kg")
    ap.add_argument("--linker", default="cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR")
    ap.add_argument("--threshold", type=float, default=0.75)
    ap.add_argument("--top_k", type=int, default=12)
    ap.add_argument("--min_seeds", type=int, default=3)
    ap.add_argument("--max_nb_per_seed", type=int, default=4)
    ap.add_argument("--max_nodes", type=int, default=48)
    ap.add_argument("--max_syn", type=int, default=12)
    ap.add_argument("--no_disease_nodes", dest="add_disease_nodes", action="store_false")
    main(ap.parse_args())


Overwriting s04_link_concepts.py


In [ ]:
%%writefile model.py
"""MediMultiBERT-KCLT model: multilingual encoder + GAT over a linked KG subgraph + fusion classifier
+ projection head for cross-lingual contrastive learning."""
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from torch_geometric.nn import GATConv
from torch_geometric.utils import softmax as pyg_softmax, scatter


class KnowledgeGAT(nn.Module):
    """Two-layer multi-head GAT over the narrative's KG subgraph, followed by text-guided
    attention pooling (the [CLS] vector is the query over node states)."""

    def __init__(self, node_dim, text_dim, hid=64, heads=4, dropout=0.1):
        super().__init__()
        self.inp = nn.Linear(node_dim, hid * heads)
        self.gat1 = GATConv(hid * heads, hid, heads=heads, dropout=dropout)
        self.gat2 = GATConv(hid * heads, hid, heads=heads, dropout=dropout)
        self.norm1, self.norm2 = nn.LayerNorm(hid * heads), nn.LayerNorm(hid * heads)
        self.query = nn.Linear(text_dim, hid * heads)
        self.drop = nn.Dropout(dropout)
        self.out_dim = hid * heads

    def forward(self, x, edge_index, batch, h_text, num_graphs):
        h = F.elu(self.inp(x))
        h = self.norm1(h + F.elu(self.gat1(self.drop(h), edge_index)))
        h = self.norm2(h + F.elu(self.gat2(self.drop(h), edge_index)))
        q = self.query(h_text)[batch]                                  # query per node from its narrative
        score = (q * h).sum(-1) / h.shape[-1] ** 0.5
        alpha = pyg_softmax(score, batch, num_nodes=h.shape[0])        # attention over each graph's nodes
        pooled = scatter(alpha[:, None] * h, batch, dim=0, dim_size=num_graphs, reduce="sum")
        return pooled, alpha


class KCLT(nn.Module):
    def __init__(self, model_name, num_classes, node_dim=768, use_kg=True, dropout=0.1,
                 gat_hid=64, gat_heads=4, proj_dim=128):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        H = self.encoder.config.hidden_size
        self.use_kg = use_kg
        self.kg = KnowledgeGAT(node_dim, H, gat_hid, gat_heads, dropout) if use_kg else None
        fused = H + (self.kg.out_dim if use_kg else 0)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(fused, 512), nn.GELU(),
                                        nn.Dropout(dropout), nn.Linear(512, num_classes))
        # projection head: used only for the contrastive loss during training
        self.proj = nn.Sequential(nn.Linear(H, H), nn.GELU(), nn.Linear(H, proj_dim))

    def forward(self, input_ids, attention_mask, graph=None):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        alpha = None
        if self.use_kg:
            g, alpha = self.kg(graph.x, graph.edge_index, graph.batch, h, num_graphs=h.shape[0])
            h_fused = torch.cat([h, g], dim=-1)
        else:
            h_fused = h
        return {"logits": self.classifier(h_fused), "z": F.normalize(self.proj(h), dim=-1),
                "h_cls": h, "node_attention": alpha}


Overwriting model.py


In [ ]:
%%writefile losses.py
"""Cross-lingual supervised contrastive loss (InfoNCE / SupCon form).

Positives for anchor i: every other sample in the batch that shares its key.
  key = source group  -> translations of the same narrative (cross-lingual alignment, default)
  key = label         -> any narrative with the same diagnosis (supervised contrastive)
Negatives: all other samples in the batch. Temperature tau.
"""
import torch


def supcon_loss(z, keys, tau=0.07):
    # z: (B, d) L2-normalised; keys: (B,) long
    B = z.shape[0]
    sim = z @ z.T / tau
    self_mask = torch.eye(B, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(self_mask, -1e9)
    pos = (keys[:, None] == keys[None, :]) & ~self_mask
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    n_pos = pos.sum(1)
    valid = n_pos > 0
    if not valid.any():
        return z.sum() * 0.0
    loss = -(log_prob * pos).sum(1)[valid] / n_pos[valid]
    return loss.mean()


Overwriting losses.py


In [ ]:
%%writefile data.py
"""Datasets, the cross-lingual batch sampler and the collate function (text + KG subgraph)."""
import random
import pandas as pd, torch
from torch.utils.data import Dataset, Sampler
from torch_geometric.data import Data, Batch


class NarrativeDataset(Dataset):
    def __init__(self, csv, label2id, graphs=None, node_emb=None):
        self.df = pd.read_csv(csv).reset_index(drop=True)
        self.y = self.df.label.map(label2id).tolist()
        gids = {g: i for i, g in enumerate(sorted(self.df.group_id.unique()))}
        self.group = self.df.group_id.map(gids).tolist()
        self.graphs, self.node_emb = graphs, node_emb

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        item = {"text": r.text, "label": self.y[i], "group": self.group[i], "uid": r.uid,
                "language": r.language, "idx": i}
        if self.graphs is not None:
            g = self.graphs[r.uid]
            item["graph"] = Data(x=self.node_emb[g["nodes"]].float(), edge_index=g["edge_index"],
                                 num_nodes=len(g["nodes"]))
        return item


class CrossLingualBatchSampler(Sampler):
    """Each batch holds (batch_size // views) source groups x `views` language versions each,
    so every sample has at least one translation of itself in the batch (contrastive positive).
    Every training row is visited once per epoch (a group's leftover row is paired with a
    random other version of the same group)."""

    def __init__(self, groups, batch_size=32, views=2, seed=0):
        self.by_group = {}
        for i, g in enumerate(groups):
            self.by_group.setdefault(g, []).append(i)
        self.bs, self.views, self.seed, self.epoch = batch_size, views, seed, 0

    def set_epoch(self, e):
        self.epoch = e

    def _units(self):
        rng = random.Random(self.seed + self.epoch)
        units = []
        for idx in self.by_group.values():
            idx = idx[:]; rng.shuffle(idx)
            for k in range(0, len(idx), self.views):
                u = idx[k:k + self.views]
                while len(u) < self.views and len(idx) > 1:
                    u.append(rng.choice([j for j in idx if j not in u] or idx))
                units.append(u)
        rng.shuffle(units)
        return units

    def __iter__(self):
        units, per = self._units(), max(1, self.bs // self.views)
        for k in range(0, len(units), per):
            yield [i for u in units[k:k + per] for i in u]

    def __len__(self):
        return (len(self._units()) + self.bs // self.views - 1) // (self.bs // self.views)


def make_collate(tokenizer, max_len=128, use_kg=True):
    def collate(items):
        enc = tokenizer([it["text"] for it in items], padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        out = {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
               "labels": torch.tensor([it["label"] for it in items]),
               "groups": torch.tensor([it["group"] for it in items]),
               "uid": [it["uid"] for it in items], "language": [it["language"] for it in items]}
        if use_kg:
            out["graph"] = Batch.from_data_list([it["graph"] for it in items])
        return out
    return collate


Overwriting data.py


In [ ]:
%%writefile train.py
"""
Train and evaluate MediMultiBERT-KCLT and its ablations / transformer baselines.

Variants
  full      XLM-R + KG-GAT + cross-lingual contrastive      (MediMultiBERT-KCLT)
  no_cl     XLM-R + KG-GAT                                  (ablation)
  no_kg     XLM-R + cross-lingual contrastive               (ablation)
  backbone  encoder + classifier only                       (XLM-R baseline; with --model_name
                                                             bert-base-multilingual-cased = mBERT baseline)

Model selection uses the VALIDATION set only (early stopping on validation loss).
The TEST set is evaluated exactly once, with the selected checkpoint.

Example
  python train.py --variant full --seed 42
  python train.py --variant backbone --model_name bert-base-multilingual-cased --tag "mBERT (Multilingual)" --seed 42
"""
import argparse, os, time, copy, json
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from common import set_seed, save_json
from data import NarrativeDataset, CrossLingualBatchSampler, make_collate
from model import KCLT
from losses import supcon_loss

VARIANTS = {"full": (True, True), "no_cl": (True, False), "no_kg": (False, True), "backbone": (False, False)}
DEFAULT_TAG = {"full": "MediMultiBERT-KCLT", "no_cl": "XLM-R + KG (no CL)",
               "no_kg": "XLM-R + CL (no KG)", "backbone": "XLM-RoBERTa (Base)"}


def to_dev(b, dev):
    out = {k: (v.to(dev) if hasattr(v, "to") else v) for k, v in b.items()}
    return out


@torch.no_grad()
def evaluate(model, loader, dev, num_classes):
    model.eval(); P, Y, L, U, A = [], [], [], [], []
    loss_sum, n, t0 = 0.0, 0, time.time()
    for b in loader:
        b = to_dev(b, dev)
        with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
            out = model(b["input_ids"], b["attention_mask"], b.get("graph"))
        logits = out["logits"].float()
        loss_sum += F.cross_entropy(logits, b["labels"], reduction="sum").item(); n += len(b["labels"])
        P.append(F.softmax(logits, -1).cpu()); Y.append(b["labels"].cpu()); L += b["language"]; U += b["uid"]
        if out["node_attention"] is not None:
            A.append((out["node_attention"].cpu(), b["graph"].batch.cpu(), len(b["labels"])))
    P = torch.cat(P).numpy(); Y = torch.cat(Y).numpy()
    return {"loss": loss_sum / n, "probs": P, "y": Y, "lang": L, "uid": U, "attn": A,
            "seconds": time.time() - t0}


def metrics(y, p, num_classes):
    yp = p.argmax(1)
    pr, rc, f1, _ = precision_recall_fscore_support(y, yp, average="macro", zero_division=0)
    onehot = np.eye(num_classes)[y]
    return {"accuracy": accuracy_score(y, yp), "precision": pr, "recall": rc, "macro_f1": f1,
            "brier": float(np.mean(np.sum((p - onehot) ** 2, 1))), "n": int(len(y)),
            "correct": int((yp == y).sum())}


def main(a):
    set_seed(a.seed)
    use_kg, use_cl = VARIANTS[a.variant]
    tag = a.tag or DEFAULT_TAG[a.variant]
    run_dir = os.path.join(a.out, tag.replace(" ", "_").replace("/", "-"), f"seed_{a.seed}")
    os.makedirs(run_dir, exist_ok=True)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    labels = sorted(pd.read_csv(f"{a.data}/train.csv").label.unique())
    label2id = {l: i for i, l in enumerate(labels)}; C = len(labels)
    graphs = node_emb = None
    if use_kg:
        graphs = torch.load(f"{a.kg}/graphs.pt"); node_emb = torch.load(f"{a.kg}/node_emb.pt")
    tok = AutoTokenizer.from_pretrained(a.model_name)
    ds = {s: NarrativeDataset(f"{a.data}/{s}.csv", label2id, graphs, node_emb) for s in ("train", "val", "test")}
    col = make_collate(tok, a.max_len, use_kg)
    sampler = CrossLingualBatchSampler(ds["train"].group, a.bs, a.views, seed=a.seed)
    dl_tr = DataLoader(ds["train"], batch_sampler=sampler, collate_fn=col, num_workers=a.workers)
    dl_va = DataLoader(ds["val"], batch_size=64, collate_fn=col, num_workers=a.workers)
    dl_te = DataLoader(ds["test"], batch_size=64, collate_fn=col, num_workers=a.workers)

    model = KCLT(a.model_name, C, node_dim=(node_emb.shape[1] if use_kg else 768), use_kg=use_kg,
                 dropout=a.dropout, gat_hid=a.gat_hid, gat_heads=a.gat_heads).to(dev)
    enc_params = list(model.encoder.parameters())
    enc_ids = {id(p) for p in enc_params}
    head_params = [p for p in model.parameters() if id(p) not in enc_ids]
    opt = torch.optim.AdamW([{"params": enc_params, "lr": a.lr},
                             {"params": head_params, "lr": a.head_lr}], weight_decay=a.wd)
    steps = len(sampler) * a.epochs
    sch = get_linear_schedule_with_warmup(opt, int(a.warmup * steps), steps)
    scaler = torch.amp.GradScaler(enabled=dev.type == "cuda")

    best, best_state, bad, log = float("inf"), None, 0, []
    for ep in range(1, a.epochs + 1):
        model.train(); sampler.set_epoch(ep); t0 = time.time()
        tl, tc, tn, tcl = 0.0, 0, 0, 0.0
        for b in dl_tr:
            b = to_dev(b, dev)
            with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                out = model(b["input_ids"], b["attention_mask"], b.get("graph"))
                ce = F.cross_entropy(out["logits"].float(), b["labels"])
                key = b["groups"] if a.cl_key == "group" else b["labels"]
                cl = supcon_loss(out["z"].float(), key, a.tau) if use_cl else torch.zeros((), device=dev)
                loss = ce + a.lam * cl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            k = len(b["labels"]); tl += ce.item() * k; tcl += cl.item() * k; tn += k
            tc += (out["logits"].argmax(-1) == b["labels"]).sum().item()
        ev = evaluate(model, dl_va, dev, C); vm = metrics(ev["y"], ev["probs"], C)
        row = {"epoch": ep, "train_ce": tl / tn, "train_contrastive": tcl / tn, "train_acc": tc / tn,
               "val_loss": ev["loss"], "val_acc": vm["accuracy"], "val_macro_f1": vm["macro_f1"],
               "epoch_seconds": time.time() - t0}
        log.append(row); print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})
        if ev["loss"] < best - 1e-4:
            best, bad = ev["loss"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = ep
        else:
            bad += 1
            if bad >= a.patience:
                print(f"early stop at epoch {ep}"); break
    pd.DataFrame(log).to_csv(f"{run_dir}/epoch_log.csv", index=False)

    # ---------------- single test evaluation with the selected checkpoint ----------------
    model.load_state_dict(best_state)
    if a.save_ckpt: torch.save(best_state, f"{run_dir}/best.pt")
    ev = evaluate(model, dl_te, dev, C); m = metrics(ev["y"], ev["probs"], C)
    per_lang = {}
    for lg in sorted(set(ev["lang"])):
        idx = [i for i, x in enumerate(ev["lang"]) if x == lg]
        per_lang[lg] = metrics(ev["y"][idx], ev["probs"][idx], C)
    rep = classification_report(ev["y"], ev["probs"].argmax(1), target_names=labels, digits=2, zero_division=0)
    n_total = sum(p.numel() for p in model.parameters())
    n_enc = sum(p.numel() for p in model.encoder.parameters())
    n_proj = sum(p.numel() for p in model.proj.parameters())
    res = {"tag": tag, "variant": a.variant, "model_name": a.model_name, "seed": a.seed,
           "best_epoch": best_epoch, "test": m, "test_per_language": per_lang,
           "params_total_M": n_total / 1e6, "params_inference_M": (n_total - n_proj) / 1e6,
           "params_added_to_encoder_M": (n_total - n_enc - n_proj) / 1e6,
           "inference_samples_per_sec": len(ev["y"]) / ev["seconds"],
           "mean_epoch_seconds": float(np.mean([r["epoch_seconds"] for r in log])),
           "hparams": vars(a)}
    save_json(res, f"{run_dir}/results.json")
    open(f"{run_dir}/classification_report.txt", "w").write(rep)

    # predictions in the format read by stat_validation.py (preds/<Model>/seed_<s>.csv)
    pdir = os.path.join(a.preds, tag); os.makedirs(pdir, exist_ok=True)
    P = pd.DataFrame(ev["probs"], columns=[f"p_{i}" for i in range(C)])
    P.insert(0, "y_pred", ev["probs"].argmax(1)); P.insert(0, "y_true", ev["y"])
    P["uid"] = ev["uid"]; P["language"] = ev["lang"]
    P.to_csv(f"{pdir}/seed_{a.seed}.csv", index=False)

    # interpretability: top-5 attended KG concepts per test narrative
    if use_kg and ev["attn"]:
        nodes = pd.read_csv(f"{a.kg}/nodes.csv").fillna("")
        rows, k = [], 0
        for alpha, batch, nb in ev["attn"]:
            for gi in range(nb):
                uid = ev["uid"][k]; k += 1
                sel = (batch == gi).nonzero().flatten()
                if len(sel) == 0:
                    rows.append({"uid": uid, "top_concepts": ""}); continue
                g_nodes = graphs[uid]["nodes"]
                order = alpha[sel].argsort(descending=True)[:5]
                rows.append({"uid": uid, "top_concepts": " | ".join(
                    f"{nodes.name[int(g_nodes[j])]} ({float(alpha[sel][j]):.2f})" for j in order)})
        pd.DataFrame(rows).to_csv(f"{run_dir}/explanations.csv", index=False)
    print(json.dumps({"tag": tag, "seed": a.seed, **m}, indent=1))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant", choices=list(VARIANTS), default="full")
    ap.add_argument("--model_name", default="xlm-roberta-base")
    ap.add_argument("--tag"); ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--data", default="data"); ap.add_argument("--kg", default="kg")
    ap.add_argument("--out", default="runs"); ap.add_argument("--preds", default="preds")
    ap.add_argument("--epochs", type=int, default=20); ap.add_argument("--patience", type=int, default=3)
    ap.add_argument("--bs", type=int, default=32); ap.add_argument("--views", type=int, default=2)
    ap.add_argument("--lr", type=float, default=2e-5); ap.add_argument("--head_lr", type=float, default=1e-4)
    ap.add_argument("--wd", type=float, default=0.01); ap.add_argument("--warmup", type=float, default=0.1)
    ap.add_argument("--tau", type=float, default=0.07); ap.add_argument("--lam", type=float, default=0.1)
    ap.add_argument("--cl_key", choices=["group", "label"], default="group")
    ap.add_argument("--dropout", type=float, default=0.1)
    ap.add_argument("--gat_hid", type=int, default=64); ap.add_argument("--gat_heads", type=int, default=4)
    ap.add_argument("--max_len", type=int, default=128); ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--save_ckpt", action="store_true")
    main(ap.parse_args())


Overwriting train.py


In [ ]:
%%writefile baselines_rnn.py
"""
Conventional baselines on the SAME multilingual split: BiLSTM + Attention, RCNN, BiGRU.
They use the XLM-R SentencePiece tokenizer (so all five languages are tokenised) with
randomly initialised 128-d embeddings. (English-only GloVe cannot represent Hindi/Tamil text,
so the earlier "GloVe + BiGRU" baseline is replaced by a subword BiGRU.)

python baselines_rnn.py --arch bilstm_att --seed 42
python baselines_rnn.py --arch rcnn --seed 42
python baselines_rnn.py --arch bigru --seed 42
"""
import argparse, os, time
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer
from common import set_seed, save_json
from train import metrics

TAGS = {"bilstm_att": "BiLSTM + Attention", "rcnn": "RCNN", "bigru": "BiGRU (subword)"}


class Net(nn.Module):
    def __init__(self, arch, vocab, C, emb=128, hid=128, drop=0.3):
        super().__init__()
        self.arch = arch
        self.emb = nn.Embedding(vocab, emb, padding_idx=1)
        if arch == "bilstm_att":
            self.rnn = nn.LSTM(emb, hid, batch_first=True, bidirectional=True); self.att = nn.Linear(2 * hid, 1); d = 2 * hid
        elif arch == "rcnn":
            self.rnn = nn.LSTM(emb, hid // 2, batch_first=True); self.conv = nn.Conv1d(hid // 2, 64, 5, padding=2); d = 64
        else:
            self.rnn = nn.GRU(emb, hid // 2, batch_first=True, bidirectional=True); d = hid
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(d, 64), nn.ReLU(), nn.Dropout(drop), nn.Linear(64, C))

    def forward(self, ids, mask):
        h, _ = self.rnn(self.emb(ids))
        m = mask.unsqueeze(-1).float()
        if self.arch == "bilstm_att":
            s = self.att(h).masked_fill(m == 0, -1e9); z = (F.softmax(s, 1) * h).sum(1)
        elif self.arch == "rcnn":
            c = F.relu(self.conv(h.transpose(1, 2))).transpose(1, 2); z = c.masked_fill(m == 0, -1e9).max(1).values
        else:
            z = (h * m).sum(1) / m.sum(1).clamp(min=1)
        return self.head(z)


def batches(df, tok, y, bs, shuffle, rng, max_len=128):
    idx = np.arange(len(df)); rng.shuffle(idx) if shuffle else None
    for k in range(0, len(idx), bs):
        j = idx[k:k + bs]
        e = tok(df.text.iloc[j].tolist(), padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        yield e["input_ids"], e["attention_mask"], torch.tensor(y[j]), j


def main(a):
    set_seed(a.seed); rng = np.random.default_rng(a.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(a.tokenizer)
    D = {s: pd.read_csv(f"{a.data}/{s}.csv") for s in ("train", "val", "test")}
    labels = sorted(D["train"].label.unique()); l2i = {l: i for i, l in enumerate(labels)}; C = len(labels)
    Y = {s: D[s].label.map(l2i).values for s in D}
    net = Net(a.arch, tok.vocab_size, C).to(dev); opt = torch.optim.Adam(net.parameters(), lr=a.lr)
    best, bad, state, log = 1e9, 0, None, []

    def run_eval(s):
        net.eval(); P = np.zeros((len(D[s]), C)); L = 0
        with torch.no_grad():
            for ids, m, y, j in batches(D[s], tok, Y[s], 128, False, rng):
                lo = net(ids.to(dev), m.to(dev)); L += F.cross_entropy(lo, y.to(dev), reduction="sum").item()
                P[j] = F.softmax(lo, -1).cpu().numpy()
        return L / len(D[s]), P

    for ep in range(1, a.epochs + 1):
        net.train(); t0 = time.time()
        for ids, m, y, _ in batches(D["train"], tok, Y["train"], a.bs, True, rng):
            loss = F.cross_entropy(net(ids.to(dev), m.to(dev)), y.to(dev))
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
        vl, vp = run_eval("val"); vm = metrics(Y["val"], vp, C)
        log.append({"epoch": ep, "val_loss": vl, "val_acc": vm["accuracy"], "epoch_seconds": time.time() - t0})
        print(log[-1])
        if vl < best - 1e-4: best, bad, state = vl, 0, {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= a.patience: break
    net.load_state_dict(state); _, tp = run_eval("test"); m = metrics(Y["test"], tp, C)
    tag = TAGS[a.arch]; rd = os.path.join(a.out, tag.replace(" ", "_"), f"seed_{a.seed}"); os.makedirs(rd, exist_ok=True)
    pd.DataFrame(log).to_csv(f"{rd}/epoch_log.csv", index=False)
    save_json({"tag": tag, "seed": a.seed, "test": m,
               "params_total_M": sum(p.numel() for p in net.parameters()) / 1e6, "hparams": vars(a)}, f"{rd}/results.json")
    pdir = os.path.join(a.preds, tag); os.makedirs(pdir, exist_ok=True)
    P = pd.DataFrame(tp, columns=[f"p_{i}" for i in range(C)]); P.insert(0, "y_pred", tp.argmax(1)); P.insert(0, "y_true", Y["test"])
    P["uid"] = D["test"].uid.values; P["language"] = D["test"].language.values
    P.to_csv(f"{pdir}/seed_{a.seed}.csv", index=False); print(tag, m)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", choices=list(TAGS), default="bilstm_att"); ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--tokenizer", default="xlm-roberta-base"); ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="runs"); ap.add_argument("--preds", default="preds")
    ap.add_argument("--epochs", type=int, default=30); ap.add_argument("--patience", type=int, default=5)
    ap.add_argument("--bs", type=int, default=32); ap.add_argument("--lr", type=float, default=1e-3)
    main(ap.parse_args())


Overwriting baselines_rnn.py


In [ ]:
%%writefile stat_validation.py
"""
Statistical validation for MediMultiBERT-KCLT (Stage 2, Reviewer Comment 2).

Takes the test-set predictions from every model and every seed, and produces:
  Table A  - mean +/- SD and 95% CI (t-distribution, across seeds) for each metric
  Table B  - significance tests of MediMultiBERT-KCLT against each baseline:
             paired t-test across seeds (accuracy, macro-F1), Cohen's d,
             exact McNemar test on each seed (same test samples), Holm-corrected
  Table C  - 95% bootstrap CI of the accuracy / macro-F1 difference (proposed minus baseline)

INPUT LAYOUT (one CSV per model per seed, all on the SAME fixed test set, same row order):
  preds/
    MediMultiBERT-KCLT/seed_42.csv  seed_101.csv  seed_202.csv  seed_303.csv  seed_404.csv
    XLM-RoBERTa (Base)/seed_42.csv  ...
    mBERT (Multilingual)/...
    BiLSTM + Attention/...  RCNN/...  GloVe + BiGRU/...
  Each CSV: columns  y_true, y_pred  and (optional, needed for Brier score)  p_0 ... p_23
  (class probabilities from softmax, in label-index order).

USAGE:  python stat_validation.py preds  "MediMultiBERT-KCLT"
Outputs: tableA_repeated_runs.csv, tableB_significance.csv, tableC_bootstrap_diff.csv
"""
import sys, os, glob
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

N_BOOT = 10000
RNG = np.random.default_rng(2026)


def load(root):
    data = {}
    for mdir in sorted(glob.glob(os.path.join(root, "*"))):
        if not os.path.isdir(mdir):
            continue
        runs = {}
        for f in sorted(glob.glob(os.path.join(mdir, "seed_*.csv"))):
            seed = os.path.basename(f)[5:-4]
            runs[seed] = pd.read_csv(f)
        if runs:
            data[os.path.basename(mdir)] = runs
    return data


def brier(df):
    pcols = [c for c in df.columns if c.startswith("p_")]
    if not pcols:
        return np.nan
    P = df[sorted(pcols, key=lambda c: int(c[2:]))].to_numpy()
    Y = np.zeros_like(P)
    Y[np.arange(len(df)), df["y_true"].to_numpy()] = 1
    return np.mean(np.sum((P - Y) ** 2, axis=1))  # multiclass Brier (sum over classes)


def metrics(df):
    yt, yp = df["y_true"], df["y_pred"]
    p, r, f, _ = precision_recall_fscore_support(yt, yp, average="macro", zero_division=0)
    return dict(Accuracy=accuracy_score(yt, yp), Precision=p, Recall=r, F1=f, Brier=brier(df))


def ci95(x):
    x = np.asarray(x, float)
    n = len(x)
    m, s = x.mean(), x.std(ddof=1)
    h = stats.t.ppf(0.975, n - 1) * s / np.sqrt(n)
    return m, s, m - h, m + h


def main(root, proposed):
    data = load(root)
    assert proposed in data, f"'{proposed}' folder not found in {root}"
    seeds = sorted(data[proposed].keys(), key=int)

    # ---------- Table A: repeated-run summary ----------
    rowsA, per_run = [], {}
    for model, runs in data.items():
        missing = set(seeds) - set(runs)
        assert not missing, f"{model} missing seeds {missing}"
        M = pd.DataFrame([metrics(runs[s]) for s in seeds], index=seeds)
        per_run[model] = M
        row = {"Model": model, "Runs": len(seeds)}
        for k in M.columns:
            if M[k].isna().all():
                row[k] = "n/a"
                continue
            m, s, lo, hi = ci95(M[k])
            row[k] = f"{m:.3f} ± {s:.3f} [{lo:.3f}, {hi:.3f}]"
        rowsA.append(row)
    A = pd.DataFrame(rowsA)
    A.to_csv("tableA_repeated_runs.csv", index=False)

    # ---------- Table B: significance tests ----------
    rowsB = []
    P = per_run[proposed]
    for model in data:
        if model == proposed:
            continue
        B = per_run[model]
        r = {"Comparison": f"{proposed} vs {model}"}
        for k in ["Accuracy", "F1"]:
            d = P[k].values - B[k].values
            t, p = stats.ttest_rel(P[k].values, B[k].values)
            r[f"Δ{k} (mean)"] = round(d.mean(), 4)
            r[f"paired t p ({k})"] = p
            r[f"Cohen d ({k})"] = round(d.mean() / d.std(ddof=1), 2) if d.std(ddof=1) > 0 else np.inf
        # exact McNemar on each seed (same test samples)
        mp = []
        for s in seeds:
            a, b = data[proposed][s], data[model][s]
            assert (a["y_true"].values == b["y_true"].values).all(), f"test rows differ for seed {s}"
            ca = a["y_pred"].values == a["y_true"].values
            cb = b["y_pred"].values == b["y_true"].values
            tbl = [[np.sum(ca & cb), np.sum(ca & ~cb)], [np.sum(~ca & cb), np.sum(~ca & ~cb)]]
            mp.append(mcnemar(tbl, exact=True).pvalue)
        r["McNemar p (max over seeds)"] = max(mp)
        r["Seeds with McNemar p<0.05"] = f"{sum(p < 0.05 for p in mp)}/{len(seeds)}"
        rowsB.append(r)
    Bt = pd.DataFrame(rowsB)
    # Holm correction across the baseline comparisons
    for col in ["paired t p (Accuracy)", "paired t p (F1)", "McNemar p (max over seeds)"]:
        Bt[col + " [Holm]"] = multipletests(Bt[col], method="holm")[1]
    Bt.to_csv("tableB_significance.csv", index=False)

    # ---------- Table C: bootstrap CI of the difference ----------
    rowsC = []
    n = len(data[proposed][seeds[0]])
    for model in data:
        if model == proposed:
            continue
        dacc, df1 = [], []
        for _ in range(N_BOOT):
            s = seeds[RNG.integers(len(seeds))]          # resample a seed
            idx = RNG.integers(0, n, n)                  # resample test samples
            a = data[proposed][s].iloc[idx]
            b = data[model][s].iloc[idx]
            ma, mb = metrics(a[["y_true", "y_pred"]]), metrics(b[["y_true", "y_pred"]])
            dacc.append(ma["Accuracy"] - mb["Accuracy"])
            df1.append(ma["F1"] - mb["F1"])
        lo_a, hi_a = np.percentile(dacc, [2.5, 97.5])
        lo_f, hi_f = np.percentile(df1, [2.5, 97.5])
        rowsC.append({"Comparison": f"{proposed} vs {model}",
                      "ΔAccuracy 95% CI": f"[{lo_a:.3f}, {hi_a:.3f}]",
                      "ΔMacro-F1 95% CI": f"[{lo_f:.3f}, {hi_f:.3f}]",
                      "CI excludes 0 (Acc)": lo_a > 0, "CI excludes 0 (F1)": lo_f > 0})
    pd.DataFrame(rowsC).to_csv("tableC_bootstrap_diff.csv", index=False)

    pd.set_option("display.width", 250, "display.max_columns", 30)
    print("\nTABLE A\n", A.to_string(index=False))
    print("\nTABLE B\n", Bt.to_string(index=False))
    print("\nTABLE C\n", pd.DataFrame(rowsC).to_string(index=False))


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else "preds",
         sys.argv[2] if len(sys.argv) > 2 else "MediMultiBERT-KCLT")


Overwriting stat_validation.py


In [ ]:
%%writefile summarise.py
"""Collect runs/*/seed_*/results.json into the manuscript tables (CSV).
  tables/main_results.csv        mean +/- SD over seeds (accuracy, P, R, macro-F1, Brier)  -> Table 12
  tables/per_language.csv        accuracy and macro-F1 per language                          -> new per-language table
  tables/complexity.csv          parameters, inference throughput, epoch time               -> Tables 13/17
  tables/epoch_log_KCLT_seed42.csv  training log of the proposed model                     -> Table 4
"""
import glob, json, os, shutil
import numpy as np, pandas as pd

os.makedirs("tables", exist_ok=True)
R = [json.load(open(f)) for f in glob.glob("runs/*/seed_*/results.json")]
if not R:
    raise SystemExit("no results found in runs/")
ms = lambda v: f"{np.mean(v):.3f} ± {np.std(v, ddof=1):.3f}" if len(v) > 1 else f"{v[0]:.3f}"

rows, lang_rows, cx = [], [], []
for tag in sorted({r["tag"] for r in R}):
    rr = [r for r in R if r["tag"] == tag]
    t = [r["test"] for r in rr]
    rows.append({"Model": tag, "Runs": len(rr), **{k: ms([x[k] for x in t]) for k in
                 ["accuracy", "precision", "recall", "macro_f1", "brier"]}})
    if "test_per_language" in rr[0]:
        for lg in rr[0]["test_per_language"]:
            lang_rows.append({"Model": tag, "Language": lg,
                              "accuracy": ms([r["test_per_language"][lg]["accuracy"] for r in rr]),
                              "macro_f1": ms([r["test_per_language"][lg]["macro_f1"] for r in rr])})
    cx.append({"Model": tag, "Params total (M)": round(rr[0].get("params_total_M", np.nan), 1),
               "Params at inference (M)": round(rr[0].get("params_inference_M", rr[0].get("params_total_M", np.nan)), 1),
               "Params added to encoder (M)": round(rr[0].get("params_added_to_encoder_M", np.nan), 2),
               "Inference (samples/s)": round(np.mean([r.get("inference_samples_per_sec", np.nan) for r in rr]), 1),
               "Epoch time (s)": round(np.mean([r.get("mean_epoch_seconds", np.nan) for r in rr]), 1)})
pd.DataFrame(rows).to_csv("tables/main_results.csv", index=False)
pd.DataFrame(lang_rows).to_csv("tables/per_language.csv", index=False)
pd.DataFrame(cx).to_csv("tables/complexity.csv", index=False)
src = "runs/MediMultiBERT-KCLT/seed_42/epoch_log.csv"
if os.path.exists(src): shutil.copy(src, "tables/epoch_log_KCLT_seed42.csv")
print(pd.DataFrame(rows).to_string(index=False)); print(pd.DataFrame(cx).to_string(index=False))


Overwriting summarise.py


In [ ]:
%%writefile requirements.txt
torch>=2.1
transformers>=4.36
torch_geometric>=2.5
sentencepiece
scikit-learn>=1.3
pandas>=2.0
numpy
scipy
statsmodels
tqdm
deep-translator


Overwriting requirements.txt


## 2. Install and download inputs

In [ ]:
!pip -q install torch_geometric deep-translator statsmodels sentencepiece sacremoses
import os
if not os.path.exists('Symptom2Disease.csv'):
    !wget -q -O Symptom2Disease.csv https://raw.githubusercontent.com/Eugeneberna/Mywork1/main/Symptom2Disease.csv
if KG_SOURCE == 'doid' and not os.path.exists('doid.obo'):
    !wget -q -O doid.obo https://raw.githubusercontent.com/DiseaseOntology/HumanDiseaseOntology/main/src/ontology/doid.obo
import pandas as pd; print(pd.read_csv('Symptom2Disease.csv').shape)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 57.3 MB/s eta 0:00:00
(1200, 3)


## 3. Translate with NLLB-200 (offline model on the GPU, about 15-25 min on a T4; cached)
After native speakers fill in `data/review/review_<lang>.csv`, run `!python s01_translate.py --apply_review --out data` and then repeat steps 4-5.

In [ ]:
if not os.path.exists('data/multilingual.csv'):
    !python s01_translate.py --input Symptom2Disease.csv --out data
print(pd.read_csv('data/multilingual.csv').language.value_counts())

language
en    1200
es    1200
fr    1200
hi    1200
ta    1200
Name: count, dtype: int64


## 4. Leakage-safe split + audit

In [ ]:
!python s02_split.py --data data/multilingual.csv --out data
print(open('data/split_audit.json').read())

sizes: {'train': 4210, 'val': 895, 'test': 895}
source_groups: {'train': 807, 'val': 173, 'test': 173}
per_language: {'train': {'en': 842, 'es': 842, 'fr': 842, 'hi': 842, 'ta': 842}, 'val': {'en': 179, 'es': 179, 'fr': 179, 'hi': 179, 'ta': 179}, 'test': {'en': 179, 'es': 179, 'fr': 179, 'hi': 179, 'ta': 179}}
test_per_class: {'Pneumonia': 45, 'Psoriasis': 40, 'Chicken pox': 40, 'Varicose Veins': 40, 'Common Cold': 40, 'Dengue': 40, 'Acne': 40, 'Impetigo': 40, 'urinary tract infection': 40, 'diabetes': 40, 'Hypertension': 40, 'Malaria': 40, 'Fungal infection': 35, 'Typhoid': 35, 'Migraine': 35, 'Arthritis': 35, 'drug reaction': 35, 'Cervical spondylosis': 35, 'Bronchial Asthma': 35, 'allergy': 35, 'peptic ulcer disease': 35, 'gastroesophageal reflux disease': 35, 'Dimorphic Hemorrhoids': 30, 'Jaundice': 30}
shared_groups: {'train&val': 0, 'train&test': 0, 'val&test': 0}
shared_source_ids: {'train&val': 0, 'train&test': 0, 'val&test': 0}
identical_texts: {'train&val': 6, 'train&test': 

## 5. Knowledge graph + concept linking
**Check `kg/seed_map.csv`**: each of the 24 diseases must map to the correct concept.

In [ ]:
if KG_SOURCE == 'umls':
    !python s03_build_kg.py --umls_dir "$UMLS_DIR" --out kg
else:
    !python s03b_doid_triples.py --obo doid.obo --out doid_triples.csv
    !python s03_build_kg.py --triples_csv doid_triples.csv --out kg
print(pd.read_csv('kg/seed_map.csv').to_string())
!python s04_link_concepts.py --data data --kg kg --out kg
print(open('kg/linking_stats.json').read())

relation
synonym        17975
is_a           17479
has_symptom     1101
Name: count, dtype: int64
{'nodes': 30744, 'edges': 36555, 'disease_nodes': 24, 'diseases_mapped': 24, 'edge_types': {'synonym': 17975, 'is_a': 17479, 'has_symptom': 1101}}
CHECK kg/seed_map.csv by hand: each of the 24 labels must map to the right concept.
                              label      cui                             name
0                         Psoriasis  N027497                        psoriasis
1                    Varicose Veins  N030424                   varicose veins
2              peptic ulcer disease  N026371             peptic ulcer disease
3                     drug reaction  N018408                     drug allergy
4   gastroesophageal reflux disease  N020014  gastroesophageal reflux disease
5                           allergy  N021259                 hypersensitivity
6           urinary tract infection  N030287          urinary tract infection
7                           Malaria  N022940   

In [ ]:
%%writefile losses.py
"""Cross-lingual supervised contrastive loss (InfoNCE / SupCon form).

Positives for anchor i: every other sample in the batch that shares its key.
  key = source group  -> translations of the same narrative (cross-lingual alignment, default)
  key = label         -> any narrative with the same diagnosis (supervised contrastive)
Negatives: all other samples in the batch. Temperature tau.
Always computed in float32 (outside autocast) for numerical stability.
"""
import torch


def supcon_loss(z, keys, tau=0.07):
    # z: (B, d) L2-normalised; keys: (B,) long
    with torch.autocast(device_type=z.device.type, enabled=False):
        z = z.float()
        B = z.shape[0]
        sim = (z @ z.T) / tau
        self_mask = torch.eye(B, dtype=torch.bool, device=z.device)
        sim = sim.masked_fill(self_mask, torch.finfo(sim.dtype).min)
        pos = (keys[:, None] == keys[None, :]) & ~self_mask
        log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
        log_prob = log_prob.masked_fill(self_mask, 0.0)
        n_pos = pos.sum(1)
        valid = n_pos > 0
        if not valid.any():
            return z.sum() * 0.0
        loss = -(log_prob * pos).sum(1)[valid] / n_pos[valid]
        return loss.mean()

Overwriting losses.py


## 6. Experiments (resumable: finished runs are skipped)

In [ ]:
import subprocess, os, sys

os.makedirs('logs', exist_ok=True)

def done(tag, seed):
    return os.path.exists(f'preds/{tag}/seed_{seed}.csv')

def run(cmd, tag, seed):
    if done(tag, seed):
        print('skip (already done):', tag, seed); return
    log_path = f"logs/{tag.replace(' ', '_').replace('/', '-')}_seed{seed}.log"
    print('\n>>>', cmd, '\n    log:', log_path, flush=True)
    lines = []
    with open(log_path, 'w') as log:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        for line in p.stdout:                       # stream output live into the notebook
            log.write(line); lines.append(line)
            if 'Warning' not in line and 'warn(' not in line:
                print(line, end='', flush=True)
        p.wait()
    if p.returncode != 0:
        print('\n' + '=' * 25 + ' ERROR (last 40 lines) ' + '=' * 25)
        print(''.join(lines[-40:]))
        raise SystemExit(f'FAILED: {cmd}  (full log: {log_path})')

seeds = [42] if MODE == 'pilot' else SEEDS
for s in seeds:
    run(f'python train.py --variant full --seed {s}', 'MediMultiBERT-KCLT', s)
    run(f'python train.py --variant backbone --seed {s}', 'XLM-RoBERTa (Base)', s)
    if MODE == 'full':
        run(f'python train.py --variant no_cl --seed {s}', 'XLM-R + KG (no CL)', s)
        run(f'python train.py --variant no_kg --seed {s}', 'XLM-R + CL (no KG)', s)
        run(f'python train.py --variant backbone --seed {s} --model_name bert-base-multilingual-cased '
            f'--tag "mBERT (Multilingual)"', 'mBERT (Multilingual)', s)
        for arch, tag in [('bilstm_att', 'BiLSTM + Attention'), ('rcnn', 'RCNN'), ('bigru', 'BiGRU (subword)')]:
            run(f'python baselines_rnn.py --arch {arch} --seed {s}', tag, s)

skip (already done): MediMultiBERT-KCLT 42
skip (already done): XLM-RoBERTa (Base) 42


## 7. Results tables

In [ ]:
!python summarise.py
if MODE == 'full':
    !python stat_validation.py preds "MediMultiBERT-KCLT"
for f in ['tables/main_results.csv','tables/per_language.csv','tables/complexity.csv']:
    if os.path.exists(f): print('\n', f); print(pd.read_csv(f).to_string(index=False))

             Model  Runs accuracy precision recall macro_f1 brier
MediMultiBERT-KCLT     1    0.950     0.953  0.951    0.949 0.091
XLM-RoBERTa (Base)     1    0.946     0.947  0.948    0.946 0.090
             Model  Params total (M)  Params at inference (M)  Params added to encoder (M)  Inference (samples/s)  Epoch time (s)
MediMultiBERT-KCLT             279.8                    279.1                         1.06                  586.1            42.2
XLM-RoBERTa (Base)             279.1                    278.4                         0.41                  619.8            39.6

 tables/main_results.csv
             Model  Runs  accuracy  precision  recall  macro_f1  brier
MediMultiBERT-KCLT     1     0.950      0.953   0.951     0.949  0.091
XLM-RoBERTa (Base)     1     0.946      0.947   0.948     0.946  0.090

 tables/per_language.csv
             Model Language  accuracy  macro_f1
MediMultiBERT-KCLT       en     0.972     0.972
MediMultiBERT-KCLT       es     0.966     0.967
Med

In [ ]:
# Bundle everything to send back for the manuscript
!zip -qr KCLT_results.zip tables runs/*/seed_*/results.json runs/*/seed_*/classification_report.txt runs/*/seed_*/epoch_log.csv runs/MediMultiBERT-KCLT/seed_42/explanations.csv data/split_audit.json data/review/review_summary.csv kg/kg_stats.json kg/linking_stats.json kg/seed_map.csv table*.csv 2>/dev/null
print('Saved', os.path.abspath('KCLT_results.zip'))

Saved /content/drive/MyDrive/KCLT/KCLT_results.zip


In [ ]:
# ====== BASELINES + ABLATIONS (resumable) ======
SEEDS = [42]            # pilot seed first (~1 hour on a T4). Then change to
                        # [42, 101, 202, 303, 404] and run again for the 5-seed statistics.

from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/KCLT')
!pip -q install torch_geometric statsmodels sentencepiece
os.makedirs('logs', exist_ok=True)

def done(tag, seed):
    return os.path.exists(f'preds/{tag}/seed_{seed}.csv')

def run(cmd, tag, seed):
    if done(tag, seed):
        print('skip (already done):', tag, seed); return
    log_path = f"logs/{tag.replace(' ', '_').replace('/', '-')}_seed{seed}.log"
    print('\n>>>', cmd, flush=True)
    lines = []
    with open(log_path, 'w') as log:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); lines.append(line)
            if line.startswith("{'epoch'") or '"accuracy"' in line or 'early stop' in line:
                print(line, end='', flush=True)
        p.wait()
    if p.returncode != 0:
        print('\n===== ERROR (last 40 lines) =====\n' + ''.join(lines[-40:]))
        raise SystemExit(f'FAILED: {cmd}')

for s in SEEDS:
    # proposed model and backbone (already done for seed 42; needed for the other seeds)
    run(f'python train.py --variant full --seed {s}',     'MediMultiBERT-KCLT', s)
    run(f'python train.py --variant backbone --seed {s}', 'XLM-RoBERTa (Base)', s)
    # ablations
    run(f'python train.py --variant no_cl --seed {s}',    'XLM-R + KG (no CL)', s)
    run(f'python train.py --variant no_kg --seed {s}',    'XLM-R + CL (no KG)', s)
    # transformer baseline
    run(f'python train.py --variant backbone --seed {s} --model_name bert-base-multilingual-cased '
        f'--tag "mBERT (Multilingual)"', 'mBERT (Multilingual)', s)
    # conventional baselines on the same five-language split
    for arch, tag in [('bilstm_att', 'BiLSTM + Attention'), ('rcnn', 'RCNN'), ('bigru', 'BiGRU (subword)')]:
        run(f'python baselines_rnn.py --arch {arch} --seed {s}', tag, s)

# ---------- tables ----------
!python summarise.py
if len(SEEDS) >= 2:                      # repeated-run statistics need at least 2 seeds
    !python stat_validation.py preds "MediMultiBERT-KCLT"

!zip -qr KCLT_results.zip tables runs preds logs data/split_audit.json kg/kg_stats.json kg/linking_stats.json kg/seed_map.csv table*.csv
print('\nDone. Send KCLT_results.zip from MyDrive/KCLT.')

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.9 MB/s eta 0:00:00
skip (already done): MediMultiBERT-KCLT 42
skip (already done): XLM-RoBERTa (Base) 42

>>> python train.py --variant no_cl --seed 42
{'epoch': 1, 'train_ce': 3.1253, 'train_contrastive': 0.0, 'train_acc': 0.0868, 'val_loss': 2.6196, 'val_acc': 0.3475, 'val_macro_f1': 0.2589, 'epoch_seconds': 42.4402}
{'epoch': 2, 'train_ce': 1.8554, 'train_contrastive': 0.0, 'train_acc': 0.519, 'val_loss': 1.0003, 'val_acc': 0.7039, 'val_macro_f1': 0.6633, 'epoch_seconds': 39.7506}
{'epoch': 3, 'train_ce': 0.6712, 'train_contrastive': 0.0, 'train_acc': 0.8412, 'val_loss': 0.353, 'val_acc': 0.9039, 'val_macro_f1': 0.9009, 'epoch_seconds': 40.8587}
{'epoch': 4, 'train_ce': 0.2347, 'train_contrastive': 0.0, 'train_acc': 0.9425, 'val_loss': 0.2752, 'val_acc': 0.9218, 'val_macro_f1': 0.9182, 'epoch_seconds': 44.8349}
{'epoch': 

In [ ]:
# ====== STAGE 1: build the knowledge graph from UMLS ======
UMLS_ZIP_URL = "https://download.nlm.nih.gov/umls/kss/2026AA/umls-2026AA-metathesaurus-full.zip"

from google.colab import drive
drive.mount('/content/drive')
import os, shutil, getpass, subprocess
os.chdir('/content/drive/MyDrive/KCLT')
!pip -q install torch_geometric statsmodels sentencepiece

# 1) download to Colab's local disk (not Drive) and keep only the three files needed
os.makedirs('/content/umls/META', exist_ok=True)
if not os.path.exists('/content/umls/META/MRREL.RRF'):
    key = getpass.getpass('UTS API key (hidden): ')
    r = subprocess.run(['curl', '-sS', '-L', '-o', '/content/umls/umls.zip',
                        f'https://uts-ws.nlm.nih.gov/download?url={UMLS_ZIP_URL}&apiKey={key}'])
    del key
    !ls -la /content/umls/umls.zip
    !unzip -o -j /content/umls/umls.zip '*MRCONSO.RRF' '*MRREL.RRF' '*MRSTY.RRF' -d /content/umls/META
    os.remove('/content/umls/umls.zip')
!ls -la /content/umls/META

# 2) keep the Disease Ontology graph and its results as a backup (nothing is deleted)
if os.path.exists('kg') and not os.path.exists('kg_doid'):
    shutil.move('kg', 'kg_doid')
os.makedirs('backup_doid/preds', exist_ok=True); os.makedirs('backup_doid/runs', exist_ok=True)
for tag in ['MediMultiBERT-KCLT', 'XLM-R + KG (no CL)']:
    for base, name in [('preds', tag), ('runs', tag.replace(' ', '_'))]:
        src = f'{base}/{name}'
        if os.path.exists(src) and not os.path.exists(f'backup_doid/{base}/{name}'):
            shutil.move(src, f'backup_doid/{base}/{name}')

# 3) build the UMLS graph and show what each disease was matched to
!python s03_build_kg.py --umls_dir /content/umls/META --out kg
import pandas as pd
print(pd.read_csv('kg/seed_map.csv').to_string())
print(open('kg/kg_stats.json').read())

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.5 MB/s eta 0:00:00
UTS API key (hidden): ··········
-rw-r--r-- 1 root root 5816418280 Oct  4 12:47 /content/umls/umls.zip
Archive:  /content/umls/umls.zip
  inflating: /content/umls/META/MRCONSO.RRF  
  inflating: /content/umls/META/MRREL.RRF  
  inflating: /content/umls/META/MRSTY.RRF  
total 8742880
drwxr-xr-x 2 root root       4096 Oct  4 12:49 .
drwxr-xr-x 3 root root       4096 Oct  4 12:49 ..
-rw------- 1 root root 2340511941 Apr  8 14:11 MRCONSO.RRF
-rw------- 1 root root 6399435346 Apr  8 14:11 MRREL.RRF
-rw------- 1 root root  212736397 Apr  8 14:11 MRSTY.RRF
[1/4] MRSTY: semantic types
[2/4] MRCONSO pass 1: locate the 24 disease concepts + symptom names
[3/4] MRREL: relations
[4/4] MRCONSO pass 2: names and ENG/SPA/FRE synonyms
{'nodes': 4381, 'edges': 8698, 'disease_nodes': 24, 'diseases_mapped': 24, 'edge_types':

In [ ]:
# ====== STAGE 2: link narratives to the UMLS graph, retrain the two graph-based models (seed 42) ======
import os, subprocess, json
import pandas as pd
os.chdir('/content/drive/MyDrive/KCLT')
os.makedirs('logs', exist_ok=True)

# 0) quick look at what the graph knows about each disease
N = pd.read_csv('kg/nodes.csv').fillna(''); E = pd.read_csv('kg/edges.csv')
name = dict(zip(N.nid, N.name)); nb = {}
for s, d in zip(E.src, E.dst):
    nb.setdefault(s, set()).add(d); nb.setdefault(d, set()).add(s)
print('Neighbours of each disease in the UMLS graph:')
for _, r in N[N.is_disease == 1].iterrows():
    ns = sorted(nb.get(r.nid, []))
    print(f"  {r.label:32s} {len(ns):4d}  e.g. {', '.join(name[i] for i in ns[:5])}")

# 1) concept linking (about 10-15 minutes on a GPU)
!python s04_link_concepts.py --data data --kg kg --out kg
print(open('kg/linking_stats.json').read())

# 2) retrain the two models that use the graph
def run(cmd, tag, seed):
    if os.path.exists(f'preds/{tag}/seed_{seed}.csv'):
        print('skip (already done):', tag, seed); return
    log_path = f"logs/{tag.replace(' ', '_')}_seed{seed}_umls.log"
    print('\n>>>', cmd, flush=True); lines = []
    with open(log_path, 'w') as log:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); lines.append(line)
            if line.startswith("{'epoch'") or '"accuracy"' in line or 'early stop' in line:
                print(line, end='', flush=True)
        p.wait()
    if p.returncode != 0:
        print('\n===== ERROR (last 40 lines) =====\n' + ''.join(lines[-40:])); raise SystemExit('FAILED')

run('python train.py --variant full --seed 42',  'MediMultiBERT-KCLT', 42)
run('python train.py --variant no_cl --seed 42', 'XLM-R + KG (no CL)', 42)

# 3) does the graph module now attend to the right disease?
ex = pd.read_csv('runs/MediMultiBERT-KCLT/seed_42/explanations.csv').fillna('')
sm = pd.read_csv('kg/seed_map.csv'); cname = dict(zip(sm.label, sm.name)); te = pd.read_csv('data/test.csv').set_index('uid')
t1 = t5 = 0
for u, tc in zip(ex.uid, ex.top_concepts):
    names = [x.rsplit(' (', 1)[0] for x in tc.split(' | ')]; tgt = cname[te.loc[u, 'label']]
    t1 += names[0] == tgt; t5 += tgt in names
print(f'\nTrue disease concept is the most-attended concept: {100*t1/len(ex):.1f}%  | in the top five: {100*t5/len(ex):.1f}%')
print('Most frequent top concepts:', ex.top_concepts.str.split(' \\(').str[0].value_counts().head(8).to_dict())

# 4) tables + results bundle (the UMLS graph itself is NOT included: it cannot be redistributed)
!python summarise.py
!rm -f KCLT_results_umls.zip
!zip -qr KCLT_results_umls.zip tables preds logs runs data/split_audit.json kg/kg_stats.json kg/linking_stats.json kg/seed_map.csv -x 'runs/*/explanations.csv'
print('\nDone. Send KCLT_results_umls.zip from MyDrive/KCLT, plus the printed output of this cell.')

Neighbours of each disease in the UMLS graph:
  Arthritis                         172  e.g. Arthritis, Gouty, Arthritis, Infectious, Arthritis, Psoriatic, Rheumatoid Arthritis, Cementitis
  Bronchial Asthma                  223  e.g. albuterol, aminophylline, Asthma, Exercise-Induced, Bronchospasm, cortisone
  Chicken pox                        34  e.g. acyclovir, vidarabine, Virus Diseases, Chickenpox with other specified complications, famciclovir
  Common Cold                       104  e.g. Acute pharyngitis, ascorbic acid, brompheniramine, Communicable Diseases, phenylpropanolamine hydrochloride
  Dengue                             18  e.g. Severe Dengue, Hemorrhagic Fevers, Viral, Virus Diseases, Flavivirus Infections, Viral disease other/nos
  diabetes                          372  e.g. Ataxia Telangiectasia, Nonproliferative diabetic retinopathy, Diabetes Mellitus, Insulin-Dependent, Lipoatrophic Diabetes Mellitus, Diabetes Mellitus, Non-Insulin-Dependent
  gastroesophageal ref

In [ ]:
# ===== FIVE-SEED RUN WITH UMLS GRAPH (resumable) =====
SEEDS = [42, 101, 202, 303, 404]

from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, shutil, subprocess
import pandas as pd
WORK = '/content/drive/MyDrive/KCLT'
os.chdir(WORK)
!pip -q install torch_geometric statsmodels sentencepiece sacremoses

# 1. Safety checks: GPU on, UMLS graph active
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU, then run again.'
kg = json.load(open('kg/kg_stats.json'))
print('GPU:', torch.cuda.get_device_name(0), '| KG nodes:', kg['nodes'], '| edges:', kg['edges'])
assert kg['nodes'] == 4381, 'The kg folder is not the UMLS graph. Stop and tell Claude.'

# 2. Models that do not use the graph: reuse seed-42 runs from the earlier backup if present
NO_KG = ['XLM-RoBERTa (Base)', 'XLM-R + CL (no KG)', 'mBERT (Multilingual)',
         'BiLSTM + Attention', 'RCNN', 'BiGRU (subword)']
for tag in NO_KG:
    for s in SEEDS:
        dst = f'preds/{tag}/seed_{s}.csv'
        src = glob.glob(f'backup_doid/**/preds/{tag}/seed_{s}.csv', recursive=True) \
            + glob.glob(f'backup_doid/preds/{tag}/seed_{s}.csv')
        if not os.path.exists(dst) and src:
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src[0], dst)
            rsrc = os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(src[0]))),
                                'runs', tag.replace(' ', '_'), f'seed_{s}')
            rdst = os.path.join('runs', tag.replace(' ', '_'), f'seed_{s}')
            if os.path.isdir(rsrc) and not os.path.isdir(rdst): shutil.copytree(rsrc, rdst)
            print('restored', tag, s)

# 3. Runner: skips finished runs, shows live output, keeps a log
os.makedirs('logs', exist_ok=True)
def run(cmd, tag, seed):
    if os.path.exists(f'preds/{tag}/seed_{seed}.csv'):
        print('skip (done):', tag, seed); return
    print('\n>>>', cmd, flush=True)
    with open(f"logs/{tag.replace(' ', '_')}_seed{seed}.log", 'w') as lf:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            lf.write(line)
            if any(k in line for k in ('epoch', 'Epoch', 'acc', 'Error', 'error', 'Traceback')):
                print(line, end='', flush=True)
        p.wait()
    if p.returncode:
        print(open(f"logs/{tag.replace(' ', '_')}_seed{seed}.log").read()[-3000:])
        raise SystemExit(f'FAILED: {cmd}')

for s in SEEDS:
    run(f'python train.py --variant full --seed {s}',     'MediMultiBERT-KCLT', s)
    run(f'python train.py --variant no_cl --seed {s}',    'XLM-R + KG (no CL)', s)
    run(f'python train.py --variant no_kg --seed {s}',    'XLM-R + CL (no KG)', s)
    run(f'python train.py --variant backbone --seed {s}', 'XLM-RoBERTa (Base)', s)
    run(f'python train.py --variant backbone --seed {s} --model_name bert-base-multilingual-cased '
        f'--tag "mBERT (Multilingual)"', 'mBERT (Multilingual)', s)
    for arch, tag in [('bilstm_att', 'BiLSTM + Attention'), ('rcnn', 'RCNN'), ('bigru', 'BiGRU (subword)')]:
        run(f'python baselines_rnn.py --arch {arch} --seed {s}', tag, s)

# 4. Tables and statistics
!python summarise.py
!python stat_validation.py preds "MediMultiBERT-KCLT"
for f in ['tables/main_results.csv', 'tables/per_language.csv', 'tableA_repeated_runs.csv',
          'tableB_significance.csv', 'tableC_bootstrap_diff.csv']:
    if os.path.exists(f): print('\n=====', f); print(pd.read_csv(f).to_string(index=False))

# 5. Versions (needed for the reproducibility comment)
import transformers, torch_geometric
print('\nVERSIONS | python', sys.version.split()[0], '| torch', torch.__version__,
      '| transformers', transformers.__version__, '| torch_geometric', torch_geometric.__version__,
      '| GPU', torch.cuda.get_device_name(0))

# 6. Results zip (no UMLS content inside)
if os.path.exists('KCLT_results_5seeds.zip'): os.remove('KCLT_results_5seeds.zip')
!zip -qr KCLT_results_5seeds.zip tables table*.csv preds runs/*/seed_*/results.json runs/*/seed_*/classification_report.txt runs/*/seed_*/epoch_log.csv data/split_audit.json kg/kg_stats.json kg/linking_stats.json
print('\nDONE. Send this file:', os.path.abspath('KCLT_results_5seeds.zip'))

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 20.9 MB/s eta 0:00:00
GPU: Tesla T4 | KG nodes: 4381 | edges: 8698
skip (done): MediMultiBERT-KCLT 42
skip (done): XLM-R + KG (no CL) 42
skip (done): XLM-R + CL (no KG) 42
skip (done): XLM-RoBERTa (Base) 42
skip (done): mBERT (Multilingual) 42
skip (done): BiLSTM + Attention 42
skip (done): RCNN 42
skip (done): BiGRU (subword) 42

>>> python train.py --variant full --seed 101
{'epoch': 1, 'train_ce': 3.1579, 'train_contrastive': 3.4506, 'train_acc': 0.0611, 'val_loss': 2.8637, 'val_acc': 0.2615, 'val_macro_f1': 0.1641, 'epoch_seconds': 40.2641}
{'epoch': 2, 'train_ce': 1.9062, 'train_contrastive': 1.253, 'train_acc': 0.518, 'val_loss': 0.8775, 'val_acc': 0.7553, 'val_macro_f1': 0.7293, 'epoch_seconds': 40.6508}
{'epoch': 3, 'train_ce': 0.6109, 'tra

In [ ]:
# ===== STEP 1: CROSS-LINGUAL TRANSFER EXPERIMENT (resumable) =====
# Labels in English only (k = 0) or English + k examples per disease per language (k = 5).
# Tested on Spanish, French, Hindi and Tamil.
SEEDS   = [42, 101, 202, 303, 404]
FEWSHOT = [0, 5]

from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, subprocess
import numpy as np, pandas as pd
os.chdir('/content/drive/MyDrive/KCLT')
subprocess.run('pip -q install torch_geometric statsmodels sentencepiece', shell=True)
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU, then run again.'
print('GPU:', torch.cuda.get_device_name(0))

open('train_xl.py', 'w').write(r'''"""
Cross-lingual transfer experiment for MediMultiBERT-KCLT.

Labels are available in ENGLISH only (zero-shot, --fewshot_k 0), or in English plus k labelled
narratives per disease in each other language (few-shot, --fewshot_k k).
  * Classification loss: labelled narratives only.
  * Contrastive loss (variants full / no_kg): each English training narrative is paired with one of
    its machine translations, which is used WITHOUT its label (unlabelled parallel text).
  * Model selection: ENGLISH validation narratives only (no target-language labels are used).
  * Test: full test set, scored once; the primary metric is accuracy on the four target languages.
Every variant sees 16 labelled English narratives per step and the same number of steps per epoch.
Translations of validation or test narratives are never used in training (grouped split).

  python train_xl.py --variant full --seed 42 --fewshot_k 0
"""
import argparse, os, time, json, random
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from common import set_seed, save_json
from data import NarrativeDataset, make_collate
from model import KCLT
from losses import supcon_loss
from train import VARIANTS, DEFAULT_TAG, to_dev, evaluate, metrics

TARGETS = ["es", "fr", "hi", "ta"]


class PairSampler(torch.utils.data.Sampler):
    """Batches of `per` units. Unit = [labelled row] or [labelled row, a translation of it]."""

    def __init__(self, df, labelled, pair, per=16, seed=0):
        self.df, self.lab, self.pair, self.per, self.seed, self.epoch = df, labelled, pair, per, seed, 0
        self.by_group = df.groupby("group_id").indices
        self.anchors = [i for i in range(len(df)) if labelled[i]]

    def set_epoch(self, e): self.epoch = e

    def __len__(self): return (len(self.anchors) + self.per - 1) // self.per

    def __iter__(self):
        rng = random.Random(self.seed * 1000 + self.epoch); lang = self.df.language.values
        units = []
        for i in self.anchors:
            u = [i]
            if self.pair:
                want_en = lang[i] != "en"      # a few-shot row is paired with its English version
                cand = [int(j) for j in self.by_group[self.df.group_id.iloc[i]]
                        if j != i and ((lang[j] == "en") == want_en)]
                if cand: u.append(rng.choice(cand))
            units.append(u)
        rng.shuffle(units)
        for k in range(0, len(units), self.per):
            yield [i for u in units[k:k + self.per] for i in u]


def fewshot_rows(df, k):
    """k labelled narratives per disease in each target language; fixed for all models and seeds."""
    if k <= 0: return set()
    rng = random.Random(42); keep = set()
    for lg in TARGETS:
        for _, g in df[df.language == lg].groupby("label"):
            idx = list(g.index); rng.shuffle(idx); keep.update(idx[:k])
    return keep


def main(a):
    set_seed(a.seed)
    use_kg, use_cl = VARIANTS[a.variant]
    tag = a.tag or DEFAULT_TAG[a.variant]
    root = a.root or f"xl_k{a.fewshot_k}"
    run_dir = os.path.join(root, "runs", tag.replace(" ", "_"), f"seed_{a.seed}"); os.makedirs(run_dir, exist_ok=True)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    labels = sorted(pd.read_csv(f"{a.data}/train.csv").label.unique()); label2id = {l: i for i, l in enumerate(labels)}; C = len(labels)
    graphs = node_emb = None
    if use_kg: graphs = torch.load(f"{a.kg}/graphs.pt"); node_emb = torch.load(f"{a.kg}/node_emb.pt")
    tok = AutoTokenizer.from_pretrained(a.model_name)
    ds = {s: NarrativeDataset(f"{a.data}/{s}.csv", label2id, graphs, node_emb) for s in ("train", "val", "test")}
    col = make_collate(tok, a.max_len, use_kg)
    df = ds["train"].df; few = fewshot_rows(df, a.fewshot_k)
    labelled = [(df.language.iloc[i] == "en") or (i in few) for i in range(len(df))]
    lab_t = torch.tensor(labelled)
    sampler = PairSampler(df, labelled, pair=use_cl, per=a.per, seed=a.seed)
    dl_tr = DataLoader(ds["train"], batch_sampler=sampler, collate_fn=col, num_workers=a.workers)
    val_en = [i for i in range(len(ds["val"])) if ds["val"].df.language.iloc[i] == "en"]
    dl_va = DataLoader(Subset(ds["val"], val_en), batch_size=64, collate_fn=col, num_workers=a.workers)
    dl_te = DataLoader(ds["test"], batch_size=64, collate_fn=col, num_workers=a.workers)
    uid2row = {u: i for i, u in enumerate(df.uid)}

    model = KCLT(a.model_name, C, node_dim=(node_emb.shape[1] if use_kg else 768), use_kg=use_kg, dropout=a.dropout).to(dev)
    enc_ids = {id(p) for p in model.encoder.parameters()}
    opt = torch.optim.AdamW([{"params": list(model.encoder.parameters()), "lr": a.lr},
                             {"params": [p for p in model.parameters() if id(p) not in enc_ids], "lr": a.head_lr}], weight_decay=a.wd)
    steps = len(sampler) * a.epochs
    sch = get_linear_schedule_with_warmup(opt, int(a.warmup * steps), steps)
    scaler = torch.amp.GradScaler(enabled=dev.type == "cuda")
    best, best_state, bad, log, best_epoch = float("inf"), None, 0, [], 0
    for ep in range(1, a.epochs + 1):
        model.train(); sampler.set_epoch(ep); t0 = time.time(); tl = tn = tc = tcl = nb = 0
        for b in dl_tr:
            m = lab_t[[uid2row[u] for u in b["uid"]]].to(dev)
            b = to_dev(b, dev)
            with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                out = model(b["input_ids"], b["attention_mask"], b.get("graph"))
                ce = F.cross_entropy(out["logits"].float()[m], b["labels"][m])
                cl = supcon_loss(out["z"].float(), b["groups"], a.tau) if use_cl else torch.zeros((), device=dev)
                loss = ce + a.lam * cl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            k = int(m.sum()); tl += ce.item() * k; tn += k; tcl += cl.item(); nb += 1
            tc += (out["logits"].argmax(-1)[m] == b["labels"][m]).sum().item()
        ev = evaluate(model, dl_va, dev, C); vm = metrics(ev["y"], ev["probs"], C)
        row = {"epoch": ep, "train_ce": tl / tn, "train_contrastive": tcl / nb, "train_acc": tc / tn,
               "val_en_loss": ev["loss"], "val_en_acc": vm["accuracy"], "epoch_seconds": time.time() - t0}
        log.append(row); print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()}, flush=True)
        if ev["loss"] < best - 1e-4:
            best, bad, best_epoch = ev["loss"], 0, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= a.patience: print(f"early stop at epoch {ep}"); break
    pd.DataFrame(log).to_csv(f"{run_dir}/epoch_log.csv", index=False)

    model.load_state_dict(best_state)
    ev = evaluate(model, dl_te, dev, C); lang = np.array(ev["lang"])
    per_lang = {lg: metrics(ev["y"][lang == lg], ev["probs"][lang == lg], C) for lg in sorted(set(lang))}
    tgt = lang != "en"; m_t = metrics(ev["y"][tgt], ev["probs"][tgt], C)
    res = {"tag": tag, "variant": a.variant, "model_name": a.model_name, "seed": a.seed, "fewshot_k": a.fewshot_k,
           "labelled_train_rows": int(sum(labelled)), "best_epoch": best_epoch, "epochs_run": len(log),
           "test_target_languages": m_t, "test_per_language": per_lang, "hparams": vars(a)}
    save_json(res, f"{run_dir}/results.json")
    P = pd.DataFrame(ev["probs"], columns=[f"p_{i}" for i in range(C)])
    P.insert(0, "y_pred", ev["probs"].argmax(1)); P.insert(0, "y_true", ev["y"]); P["uid"] = ev["uid"]; P["language"] = ev["lang"]
    for sub, mask in (("preds", tgt), ("preds_all", np.ones(len(P), bool))):   # preds = target languages only
        d = os.path.join(root, sub, tag); os.makedirs(d, exist_ok=True); P[mask].to_csv(f"{d}/seed_{a.seed}.csv", index=False)
    print(json.dumps({"tag": tag, "seed": a.seed, "k": a.fewshot_k, "target_accuracy": m_t["accuracy"],
                      **{lg: round(v["accuracy"], 4) for lg, v in per_lang.items()}}))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant", choices=list(VARIANTS), default="full")
    ap.add_argument("--model_name", default="xlm-roberta-base"); ap.add_argument("--tag")
    ap.add_argument("--seed", type=int, default=42); ap.add_argument("--fewshot_k", type=int, default=0)
    ap.add_argument("--data", default="data"); ap.add_argument("--kg", default="kg"); ap.add_argument("--root")
    ap.add_argument("--epochs", type=int, default=30); ap.add_argument("--patience", type=int, default=3)
    ap.add_argument("--per", type=int, default=16)
    ap.add_argument("--lr", type=float, default=2e-5); ap.add_argument("--head_lr", type=float, default=1e-4)
    ap.add_argument("--wd", type=float, default=0.01); ap.add_argument("--warmup", type=float, default=0.1)
    ap.add_argument("--tau", type=float, default=0.07); ap.add_argument("--lam", type=float, default=0.1)
    ap.add_argument("--dropout", type=float, default=0.1); ap.add_argument("--max_len", type=int, default=128)
    ap.add_argument("--workers", type=int, default=2)
    main(ap.parse_args())
''')
open('kg_check.py', 'w').write(r'''"""Does the knowledge graph carry symptom-to-disease knowledge? Uses the graph ALONE (no training):
for each narrative, the predicted disease is the disease node with most edges to the linked concepts."""
import pandas as pd, torch, numpy as np, sys
kg, data = (sys.argv[1:3] + ["kg", "data"])[:2] if len(sys.argv) > 2 else ("kg", "data")
N = pd.read_csv(f"{kg}/nodes.csv").fillna(""); dis = dict(zip(N[N.is_disease == 1].nid, N[N.is_disease == 1].label))
G = torch.load(f"{kg}/graphs.pt"); T = pd.read_csv(f"{data}/train.csv"); rows = []
for uid, lab, lg in zip(T.uid, T.label, T.language):
    g = G[uid]; nodes = g["nodes"].tolist(); cnt = {d: 0 for d in dis.values()}
    for s, d in g["edge_index"].T.tolist():
        if nodes[s] in dis and nodes[d] not in dis: cnt[dis[nodes[s]]] += 1
    top = max(cnt.values()); best = [k for k, v in cnt.items() if v == top]
    rows.append({"language": lg, "true disease linked": cnt[lab] > 0, "graph-only correct": top > 0 and len(best) == 1 and best[0] == lab})
R = pd.DataFrame(rows); out = (100 * R.groupby("language").mean()).round(1); out.loc["all"] = (100 * R.drop(columns="language").mean()).round(1)
print("KNOWLEDGE-GRAPH CHECK on training narratives (%; chance for 24 diseases = 4.2%)"); print(out.to_string())
''')

# 1. Does the graph hold symptom-to-disease knowledge? (no training, takes seconds)
subprocess.run('python kg_check.py', shell=True)

# 2. Runs (finished runs are skipped)
os.makedirs('logs', exist_ok=True)
def run(cmd, root, tag, seed):
    if os.path.exists(f'{root}/preds/{tag}/seed_{seed}.csv'):
        print('skip (done):', root, tag, seed); return
    print('\n>>>', cmd, flush=True)
    lf = f"logs/{root}_{tag.replace(' ', '_')}_seed{seed}.log"
    with open(lf, 'w') as f:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line)
            if any(s in line for s in ('early stop', 'target_accuracy', 'Error', 'Traceback')): print(line, end='', flush=True)
        p.wait()
    if p.returncode:
        print(open(lf).read()[-3000:]); raise SystemExit(f'FAILED: {cmd}')

JOBS = [('full', 'MediMultiBERT-KCLT', ''), ('backbone', 'XLM-RoBERTa (Base)', ''),
        ('no_cl', 'XLM-R + KG (no CL)', ''), ('no_kg', 'XLM-R + CL (no KG)', ''),
        ('backbone', 'mBERT (Multilingual)', ' --model_name bert-base-multilingual-cased --tag "mBERT (Multilingual)"')]
for k in FEWSHOT:
    for s in SEEDS:
        for variant, tag, extra in JOBS:
            run(f'python train_xl.py --variant {variant} --seed {s} --fewshot_k {k}{extra}', f'xl_k{k}', tag, s)

# 3. Results: accuracy on the four target languages, mean +/- SD over seeds, and paired tests
LG = ['en', 'es', 'fr', 'hi', 'ta']
for k in FEWSHOT:
    root = f'xl_k{k}'; rows = []
    for tag in [j[1] for j in JOBS]:
        R = [json.load(open(f)) for f in sorted(glob.glob(f"{root}/runs/{tag.replace(' ', '_')}/seed_*/results.json"))]
        t = [100 * r['test_target_languages']['accuracy'] for r in R]
        row = {'Model': tag, 'Runs': len(R), 'Target languages (es, fr, hi, ta)': f'{np.mean(t):.1f} ± {np.std(t, ddof=1):.1f}'}
        for l in LG: row[l] = f"{np.mean([100 * r['test_per_language'][l]['accuracy'] for r in R]):.1f}"
        rows.append(row)
    print(f'\n===== k = {k}: test accuracy (%), labels in ' + ('English only' if k == 0 else f'English + {k} per disease per language') + ' =====')
    T = pd.DataFrame(rows); print(T.to_string(index=False)); T.to_csv(f'{root}/summary.csv', index=False)
    subprocess.run(f'cd {root} && python ../stat_validation.py preds "MediMultiBERT-KCLT" > stats_log.txt 2>&1', shell=True)
    for f in ['tableB_significance.csv', 'tableC_bootstrap_diff.csv']:
        if os.path.exists(f'{root}/{f}'): print('\n', f); print(pd.read_csv(f'{root}/{f}').to_string(index=False))
        else: print('statistics not produced, see', f'{root}/stats_log.txt')

# 4. Results zip (no UMLS content inside)
if os.path.exists('KCLT_results_transfer.zip'): os.remove('KCLT_results_transfer.zip')
subprocess.run('zip -qr KCLT_results_transfer.zip xl_k*/preds xl_k*/preds_all xl_k*/*.csv xl_k*/stats_log.txt xl_k*/runs/*/seed_*/results.json xl_k*/runs/*/seed_*/epoch_log.csv', shell=True)
print('\nDONE. Send this file:', os.path.abspath('KCLT_results_transfer.zip'))

Mounted at /content/drive
GPU: Tesla T4

>>> python train_xl.py --variant full --seed 42 --fewshot_k 0
early stop at epoch 15
{"tag": "MediMultiBERT-KCLT", "seed": 42, "k": 0, "target_accuracy": 0.8379888268156425, "en": 0.9721, "es": 0.905, "fr": 0.8436, "hi": 0.8492, "ta": 0.7542}

>>> python train_xl.py --variant backbone --seed 42 --fewshot_k 0
early stop at epoch 14
{"tag": "XLM-RoBERTa (Base)", "seed": 42, "k": 0, "target_accuracy": 0.5432960893854749, "en": 0.9888, "es": 0.6648, "fr": 0.648, "hi": 0.5084, "ta": 0.352}

>>> python train_xl.py --variant no_cl --seed 42 --fewshot_k 0
early stop at epoch 14
{"tag": "XLM-R + KG (no CL)", "seed": 42, "k": 0, "target_accuracy": 0.49162011173184356, "en": 0.9609, "es": 0.6034, "fr": 0.5922, "hi": 0.4581, "ta": 0.3128}

>>> python train_xl.py --variant no_kg --seed 42 --fewshot_k 0
early stop at epoch 16
{"tag": "XLM-R + CL (no KG)", "seed": 42, "k": 0, "target_accuracy": 0.86731843575419, "en": 0.9665, "es": 0.9106, "fr": 0.8603, "hi": 